[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dwzhu-pku/PaperBanana/blob/main/notebooks/PaperBanana_Colab_End_to_End.ipynb)

# PaperBanana End-to-End Colab Notebook

Simple end-to-end workflow: paste your Google API key + run in Colab. You can upload a PDF/text/data file, or paste text directly, then generate publication-style figures with the PaperBanana pipeline.

Project links:
- Project website: https://paper-banana.org/
- Main repository: https://github.com/dwzhu-pku/PaperBanana
- Main paper (DOI): https://doi.org/10.48550/arXiv.2601.23265
- Hugging Face dataset page: https://huggingface.co/datasets/dwzhu/PaperBananaBench
- Original version under Google-Research: https://github.com/google-research/papervizagent

## 10-Step Roadmap

1. Mount Google Drive
2. Clone/update PaperBanana in Drive
3. Install dependencies and create required folders
4. Configure API key and model settings
5. Choose your input source/file
6. Set core generation settings
7. (Optional) tune advanced settings
8. Build generation inputs from your paper/text
9. Run generation (with cost estimate) and save outputs
10. (Optional) inspect intermediate stages

In [ ]:
#@title Step 1: Mount Google Drive and define project paths
from pathlib import Path
import os
from google.colab import drive

# Mount Drive (configurable via PAPERBANANA_DRIVE_MOUNT_POINT)
DRIVE_MOUNT_POINT = Path(os.environ.get('PAPERBANANA_DRIVE_MOUNT_POINT', str(Path.home() / 'drive')))
drive.mount(str(DRIVE_MOUNT_POINT))

# All notebook artifacts are stored under this Drive folder
DRIVE_ROOT = DRIVE_MOUNT_POINT / 'MyDrive'
PB_DRIVE_ROOT = DRIVE_ROOT / 'PaperBanana'
REPO_DIR = PB_DRIVE_ROOT / 'PaperBanana'
INPUT_DIR = PB_DRIVE_ROOT / 'input'
OUTPUT_DIR = PB_DRIVE_ROOT / 'outputs'

PB_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drive root: {DRIVE_ROOT}')
print(f'Project root in Drive: {PB_DRIVE_ROOT}')
print(f'Repo path: {REPO_DIR}')
print(f'Input path: {INPUT_DIR}')
print(f'Output path: {OUTPUT_DIR}')
print('Upload your paper to the input folder before Step 5.')


In [ ]:
#@title Step 2: Clone repo to Drive (or update if already cloned)
import subprocess

GITHUB_REPO_URL = 'https://github.com/dwzhu-pku/PaperBanana.git'  # @param {type:"string"}
# Keep True in most cases.
# - True: if the repo folder already exists, fetch/pull latest updates from GitHub.
# - False: keep the current local repo copy exactly as-is (no update).
UPDATE_EXISTING_REPO = True  # @param {type:"boolean"}

if UPDATE_EXISTING_REPO:
    print('UPDATE_EXISTING_REPO=True: existing Drive clone will be updated from GitHub.')
else:
    print('UPDATE_EXISTING_REPO=False: existing Drive clone will be reused without update.')


def run_cmd(cmd):
    print('>', ' '.join(cmd))
    subprocess.run(cmd, check=True)


if not (REPO_DIR / '.git').exists():
    PB_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    run_cmd(['git', 'clone', GITHUB_REPO_URL, str(REPO_DIR)])
    print('Repository cloned successfully.')
else:
    print('Repository already exists in Drive.')
    if UPDATE_EXISTING_REPO:
        run_cmd(['git', '-C', str(REPO_DIR), 'fetch', '--all'])
        # Pull current checked-out branch; if this fails, continue with local copy.
        try:
            run_cmd(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
            print('Repository updated.')
        except subprocess.CalledProcessError:
            print('Could not fast-forward pull. Keeping current local branch state.')

print('Repo ready:', REPO_DIR)


In [ ]:
#@title Step 3: Install dependencies and create required placeholder files
import os
import sys
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Working directory:', Path.cwd())
print('Python executable:', sys.executable)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(REPO_DIR / 'requirements.txt'),
    'pypdf',
    'ipyfilechooser'
], check=True)

# PaperBanana planner expects these ref.json files to exist even when retrieval_setting='none'.
for task in ['diagram', 'plot']:
    ref_path = REPO_DIR / 'data' / 'PaperBananaBench' / task / 'ref.json'
    ref_path.parent.mkdir(parents=True, exist_ok=True)
    if not ref_path.exists():
        ref_path.write_text('[]\n', encoding='utf-8')

print('Dependencies installed and placeholder data files are ready.')
print()
print('If this is your FIRST run on a fresh runtime, restart now:')
print('  Runtime -> Restart runtime, then re-run from Step 1.')
print('If you already restarted once, ignore this message.')


## Step 4: API Key and Model Setup

For first run: set your API key, keep the recommended defaults, and run the Step 4 code cell.

| Field | Recommended for first run | What it controls / when to change |
|---|---|---|
| `GOOGLE_API_KEY_INPUT` | Paste your Gemini API key | Required for all Gemini calls in this notebook. |
| `OPENAI_API_KEY_INPUT` | Leave empty unless needed | Optional. Needed when using OpenAI-compatible paths (for example `gpt-image-*` custom image model). |
| `ANTHROPIC_API_KEY_INPUT` | Leave empty unless needed | Optional. Reserved for compatible paths; main notebook text pipeline remains Gemini-based. |
| `STORE_API_KEY_IN_CONFIG` | `False` | Security default. `False` keeps key in runtime env only (not written to Drive). |
| `VALIDATE_API_KEY_NOW` | `True` | Keeps a small key-check call enabled. Set `False` only to skip validation. |
| `VALIDATION_MODEL` | `gemini-2.5-flash` | Used only for the API-key validation call. |
| `TEXT_MODEL_PRESET` | `fast_2_5_flash` | Text reasoning pipeline (Step 8 extraction + planner/stylist/critic text). |
| `IMAGE_MODEL_PRESET` | `nano_banana_2` | Image rendering/refinement pipeline. |
| `CUSTOM_TEXT_MODEL` | Leave empty | Use only when `TEXT_MODEL_PRESET='custom'`. |
| `CUSTOM_IMAGE_MODEL` | Leave empty | Use only when `IMAGE_MODEL_PRESET='custom'`. |

Why `fast_2_5_flash` is the default in this notebook:
- Reliability-first for Colab first runs (fewer timeout/503 retries in Steps 8-9).
- Repo code does not hardcode one universal text-model default; it reads from `configs/model_config.yaml` / environment.

Preset tradeoffs:
- Text: `fast_2_5_flash` (most stable), `balanced_2_5_pro` (better quality, slower), `paper_style_updated` (highest quality, highest cost).
- Image: `nano_banana_2` (recommended), `nano_banana_pro` (highest quality/cost), `nano_banana` (fastest/cheapest).

Pricing reference used by Step 9 estimator:
- https://ai.google.dev/gemini-api/docs/pricing


In [ ]:
#@title Step 4: API key and model selection

# Paste your Gemini API key in this box.
GOOGLE_API_KEY_INPUT = ''  # @param {type:"string"}
# Optional: for OpenAI-compatible paths (e.g., custom image model like gpt-image-1.5).
OPENAI_API_KEY_INPUT = ''  # @param {type:"string"}
# Optional: loaded to runtime env for compatible custom branches; not used in the default text pipeline.
ANTHROPIC_API_KEY_INPUT = ''  # @param {type:"string"}

# If True, run a tiny test request first so invalid keys fail fast.
VALIDATE_API_KEY_NOW = True  # @param {type:"boolean"}
# Model used only for the key-check request above.
VALIDATION_MODEL = 'gemini-2.5-flash'  # @param {type:"string"}

# Main text-model preset for extraction/planner/critic/stylist text reasoning.
TEXT_MODEL_PRESET = 'fast_2_5_flash'  # @param ['paper_style_updated', 'balanced_2_5_pro', 'fast_2_5_flash', 'custom']
# Used only when TEXT_MODEL_PRESET='custom'.
CUSTOM_TEXT_MODEL = ''  # @param {type:"string"}

# Main image-model preset for final rendering and optional refinement.
IMAGE_MODEL_PRESET = 'nano_banana_2'  # @param ['nano_banana_2', 'nano_banana_pro', 'nano_banana', 'custom']
# Used only when IMAGE_MODEL_PRESET='custom'.
CUSTOM_IMAGE_MODEL = ''  # @param {type:"string"}

# Security default: keep API key in runtime env only (not written to Drive config file).
STORE_API_KEY_IN_CONFIG = False  # @param {type:"boolean"}

import os
import yaml

TEXT_MODEL_MAP = {
    'paper_style_updated': 'gemini-3.1-pro-preview',
    'balanced_2_5_pro': 'gemini-2.5-pro',
    'fast_2_5_flash': 'gemini-2.5-flash',
}

IMAGE_MODEL_MAP = {
    'nano_banana_2': 'gemini-3.1-flash-image-preview',
    'nano_banana_pro': 'gemini-3-pro-image-preview',
    'nano_banana': 'gemini-2.5-flash-image',
}


def resolve_model(preset: str, custom: str, mapping: dict) -> str:
    if preset == 'custom':
        model = (custom or '').strip()
    else:
        model = mapping.get(preset, '').strip()
    if not model:
        raise ValueError('Model selection is empty. Choose a preset or provide CUSTOM_*_MODEL.')
    return model


MODEL_NAME = resolve_model(TEXT_MODEL_PRESET, CUSTOM_TEXT_MODEL, TEXT_MODEL_MAP)
IMAGE_MODEL_NAME = resolve_model(IMAGE_MODEL_PRESET, CUSTOM_IMAGE_MODEL, IMAGE_MODEL_MAP)

if 'gemini' not in MODEL_NAME.lower():
    raise ValueError(
        f'TEXT model "{MODEL_NAME}" is not supported in this notebook flow. '
        'Retriever/Planner/Stylist/Critic are Gemini-based here. '
        'Use a Gemini text model in Step 4. '
        'OPENAI/ANTHROPIC keys are optional for compatible custom branches.'
    )

# Supports both the box above and `%env GOOGLE_API_KEY=...` in Colab.
api_key = (GOOGLE_API_KEY_INPUT or os.environ.get('GOOGLE_API_KEY', '')).strip()
if not api_key:
    raise ValueError(
        'API key is empty. Get a key at https://aistudio.google.com/apikey , paste it into GOOGLE_API_KEY_INPUT, and re-run Step 4.'
    )

# Make key available to repo code.
os.environ['GOOGLE_API_KEY'] = api_key

# Optional API keys for compatible paths.
openai_api_key = (OPENAI_API_KEY_INPUT or os.environ.get('OPENAI_API_KEY', '')).strip()
anthropic_api_key = (ANTHROPIC_API_KEY_INPUT or os.environ.get('ANTHROPIC_API_KEY', '')).strip()
if openai_api_key:
    os.environ['OPENAI_API_KEY'] = openai_api_key
if anthropic_api_key:
    os.environ['ANTHROPIC_API_KEY'] = anthropic_api_key

if IMAGE_MODEL_PRESET == 'custom':
    if ('image' not in IMAGE_MODEL_NAME.lower()) and ('nanoviz' not in IMAGE_MODEL_NAME.lower()):
        print(
            f'Warning: IMAGE_MODEL_NAME="{IMAGE_MODEL_NAME}" does not contain "image" or "nanoviz". '
            'Verify this is a valid image generation model ID.'
        )

# Optional validation only when requested.
if VALIDATE_API_KEY_NOW:
    from google import genai
    from google.genai import types

    try:
        client = genai.Client(api_key=api_key)
        _ = client.models.generate_content(
            model=VALIDATION_MODEL,
            contents='ping',
            config=types.GenerateContentConfig(max_output_tokens=4, temperature=0),
        )
        print('API key validation: OK')
    except Exception as e:
        msg = str(e)
        if ('API_KEY_INVALID' in msg) or ('API key not valid' in msg):
            raise ValueError(
                'GOOGLE_API_KEY is invalid for Gemini API. '
                'Create/copy a valid key at https://aistudio.google.com/apikey and paste it in GOOGLE_API_KEY_INPUT.'
            ) from e
        raise ValueError(f'API key validation failed: {e}') from e
else:
    print('API key validation skipped (fast mode).')

config_payload = {
    'defaults': {
        'model_name': MODEL_NAME,
        'image_model_name': IMAGE_MODEL_NAME,
    },
    'api_keys': {
        'google_api_key': (api_key if STORE_API_KEY_IN_CONFIG else ''),
        'openai_api_key': (openai_api_key if STORE_API_KEY_IN_CONFIG else ''),
        'anthropic_api_key': (anthropic_api_key if STORE_API_KEY_IN_CONFIG else ''),
    },
}

config_path = REPO_DIR / 'configs' / 'model_config.yaml'
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config_payload, f, sort_keys=False)

print('Saved config to:', config_path)
print('MODEL_NAME:', MODEL_NAME)
print('IMAGE_MODEL_NAME:', IMAGE_MODEL_NAME)
if STORE_API_KEY_IN_CONFIG:
    print('Warning: API key(s) persisted to config file on Drive. Keep the folder private.')
else:
    print('Security mode: API key(s) kept in runtime env only (not persisted to Drive config).')
if openai_api_key:
    print('OPENAI_API_KEY loaded in runtime environment (used only for compatible custom branches, e.g., gpt-image-* image models).')
if anthropic_api_key:
    print('ANTHROPIC_API_KEY loaded in runtime environment (compatibility only; default notebook path does not call Anthropic text models).')
print('Tip: You can set key via magic command in Colab: %env GOOGLE_API_KEY=your_key')


## Input Options

Use Step 5 to choose one input file path source:
- `upload_from_computer`: upload from your local machine to Drive.
- `use_drive_path`: type/paste an existing Drive path.
- `browse_drive`: pick a file with a browser widget.

Then choose in Step 6 how to build inputs:
- `drive_pdf_auto`: read PDF and auto-extract context/intent/caption.
- `paste_text_manual`: paste text directly and skip extraction.
- `drive_file_manual`: read a `.txt/.md/.tex/.pdf/.csv/.json` from Drive.
- `paste_plot_data_manual`: paste plot raw data directly.


In [ ]:
#@title Step 5: Choose input file (upload or Drive path)
from pathlib import Path
from IPython.display import display, Markdown

FILE_INPUT_METHOD = 'upload_from_computer'  # @param ['upload_from_computer', 'use_drive_path', 'browse_drive']
FALLBACK_SELECTED_PATH = str(INPUT_DIR / 'paper.pdf')  # @param {type:"string"}
BROWSER_ROOT = str(INPUT_DIR)  # @param {type:"string"}

# Step 8 will read this variable.
SELECTED_DRIVE_FILE = ''
USE_WIDGET_SELECTION = True

ALLOWED_PATTERNS = ['*.pdf', '*.txt', '*.md', '*.tex', '*.csv', '*.json', '*.tsv']


def _set_selected(path_text: str):
    global SELECTED_DRIVE_FILE
    path = Path(str(path_text or '').strip()).expanduser()
    if path.exists() and path.is_file():
        SELECTED_DRIVE_FILE = str(path)
        print('Selected file:', SELECTED_DRIVE_FILE)
        return True
    return False


if FILE_INPUT_METHOD == 'upload_from_computer':
    try:
        from google.colab import files

        print('Click "Choose Files" and upload your paper/data file from your computer.')
        uploaded = files.upload()

        if not uploaded:
            print('No file uploaded. You can re-run Step 5.')
        else:
            first_name = list(uploaded.keys())[0]
            target_path = INPUT_DIR / first_name
            target_path.write_bytes(uploaded[first_name])
            SELECTED_DRIVE_FILE = str(target_path)
            FALLBACK_SELECTED_PATH = str(target_path)
            print('Uploaded to Drive:', target_path)
            display(Markdown('**Next:** run **Step 6** then **Step 8**.'))

    except Exception as e:
        print('Upload widget failed:', e)
        print('Use FILE_INPUT_METHOD=use_drive_path and set FALLBACK_SELECTED_PATH.')

elif FILE_INPUT_METHOD == 'use_drive_path':
    if _set_selected(FALLBACK_SELECTED_PATH):
        print('Using FALLBACK_SELECTED_PATH.')
        display(Markdown('**Next:** run **Step 6** then **Step 8**.'))
    else:
        print('Path not found:', FALLBACK_SELECTED_PATH)
        print('Update FALLBACK_SELECTED_PATH and re-run Step 5.')

elif FILE_INPUT_METHOD == 'browse_drive':
    try:
        import ipywidgets as widgets
        from ipyfilechooser import FileChooser

        root = Path(BROWSER_ROOT)
        if not root.exists():
            root = INPUT_DIR

        chooser = FileChooser(str(root))
        chooser.title = '<b>Select a file, then click Use selected file</b>'
        chooser.show_hidden = False
        chooser.use_dir_icons = True
        chooser.filter_pattern = ALLOWED_PATTERNS

        # Keep legacy name for compatibility with Step 8.
        pdf_chooser = chooser

        use_btn = widgets.Button(description='Use selected file', button_style='success')
        out = widgets.Output()

        def on_use_clicked(_):
            chosen = str(getattr(chooser, 'selected', '') or '').strip()
            with out:
                out.clear_output()
                if not chosen:
                    print('No file selected yet.')
                    return
                if _set_selected(chosen):
                    display(Markdown('**Next:** run **Step 6** then **Step 8**.'))
                else:
                    print('Selected path is invalid:', chosen)

        use_btn.on_click(on_use_clicked)
        display(chooser, use_btn, out)

    except Exception as e:
        print('Browser widget unavailable:', e)
        print('Use FILE_INPUT_METHOD=use_drive_path and set FALLBACK_SELECTED_PATH.')

else:
    raise ValueError(f'Unknown FILE_INPUT_METHOD: {FILE_INPUT_METHOD}')


## Core Settings (Step 6)

Use the Step 6 code cell fields below with these meanings:

| Parameter | What it controls |
|---|---|
| `UI_MODE` | `simple` = one result. `advanced` = multiple candidates + refinement options. |
| `TASK_NAME` | `diagram` for method diagrams, `plot` for statistical plots. |
| `INPUT_MODE` | Where source text/data comes from (auto PDF, manual text, manual file, pasted plot data). |
| `PDF_PAGE_RANGE` | Limits PDF pages (example: `3-8` or `5`). Empty = all pages. |
| `CONTEXT_SCOPE` | `auto_method_first` (default) tries method-first then full-paper fallback, `method_only`, or `full_paper`. |
| `MUST_INCLUDE_TERMS` | Comma-separated critical terms that Step 8 will try to enforce in outputs. |
| `STYLE_MODE` | `planner_critic_fast` (default, Streamlit `demo_planner_critic`) or `stylist_enhanced` (higher quality, slower, Streamlit `demo_full`). |
| `VISUAL_INTENT_HINT` | One-line guidance for what the figure should communicate. |
| `MANUAL_FIGURE_CAPTION` | Optional explicit caption. If empty, notebook auto-generates one. |
| `PASTED_METHOD_TEXT` | Used only when `INPUT_MODE='paste_text_manual'`. |
| `PLOT_VISUAL_INTENT` | Used for plot task to describe the desired chart. |
| `PLOT_RAW_DATA_FILE_PATH` | File path used when `TASK_NAME='plot'` and `INPUT_MODE='drive_file_manual'`. |
| `PASTED_PLOT_RAW_DATA` | Used when `INPUT_MODE='paste_plot_data_manual'`. |


Special-concept tip:
- If your figure must include a specific concept (for example: **a two-branch control vs treatment workflow with explicit sample transfer**), set `CONTEXT_SCOPE='full_paper'` and set `MUST_INCLUDE_TERMS` (example: `control group, treatment group, sample transfer`).


In [ ]:
#@title Step 6: Core settings (simple mode)
UI_MODE = 'simple'  # @param ['simple', 'advanced']
TASK_NAME = 'diagram'  # @param ['diagram', 'plot']

INPUT_MODE = 'drive_pdf_auto'  # @param ['drive_pdf_auto', 'paste_text_manual', 'drive_file_manual', 'paste_plot_data_manual']
PDF_PAGE_RANGE = ''  # @param {type:"string"}

# Extraction scope for drive_pdf_auto:
# auto_method_first: try method-only extraction first; auto-fallback to full-paper if coverage is low.
# method_only: use method-focused sections only.
# full_paper: use full paper with chunked coverage extraction.
CONTEXT_SCOPE = 'auto_method_first'  # @param ['auto_method_first', 'method_only', 'full_paper']

# Optional comma-separated terms that must be preserved in Step 8 outputs.
# Use this for must-keep concepts (example: control group, treatment group, sample transfer).
MUST_INCLUDE_TERMS = ''  # @param {type:"string"}

# Style/pipeline choice from Streamlit app behavior:
STYLE_MODE = 'planner_critic_fast'  # @param ['planner_critic_fast', 'stylist_enhanced']

# Used in diagram mode.
VISUAL_INTENT_HINT = 'Create a clear methodology overview diagram of the main contribution.'  # @param {type:"string"}
MANUAL_FIGURE_CAPTION = ''  # @param {type:"string"}
PASTED_METHOD_TEXT = '''
'''

# Used in plot mode.
PLOT_VISUAL_INTENT = 'Create a publication-quality plot showing the main result trend with clear axes and legend.'  # @param {type:"string"}
PLOT_RAW_DATA_FILE_PATH = str(INPUT_DIR / 'raw_data.csv')  # @param {type:"string"}
PASTED_PLOT_RAW_DATA = '''
'''

# Safe defaults (used unless Step 7 overrides them).
USE_WIDGET_SELECTION = True
AUTO_EXTRACT_MAX_ATTEMPTS = 3
STEP8_FALLBACK_MODELS = 'gemini-2.5-pro,gemini-3.1-pro-preview'
ASPECT_RATIO = '16:9'
if STYLE_MODE == 'stylist_enhanced':
    PIPELINE_MODE = 'demo_full'
else:
    PIPELINE_MODE = 'demo_planner_critic'
RETRIEVAL_SETTING = 'none'
MAX_CRITIC_ROUNDS = 3
NUM_CANDIDATES = 10
MAX_CONCURRENT = 10
SELECTED_CANDIDATE_INDEX = 0
RUN_REFINEMENT = False
REFINE_SOURCE = 'selected_result'
EXTERNAL_REFINE_IMAGE_PATH = ''
REFINE_PROMPT = 'Keep everything the same but output a cleaner higher-resolution publication-quality figure.'
REFINE_RESOLUTION = '2K'
REFINE_ASPECT_RATIO = '16:9'
EXPORT_4K = True
ZIP_EXPORT = True
SHOW_STAGE_DESCRIPTIONS = True
SHOW_CRITIC_SUGGESTIONS = True
STEP8_OUTER_MAX_ATTEMPTS = 4
STEP8_PER_ATTEMPT_TIMEOUT = 60
STEP8_MAX_PAYLOAD_CHARS = 50000

# Step 8 chunk/coverage defaults.
CHUNK_SIZE_CHARS = 9000
CHUNK_OVERLAP_CHARS = 1200
STEP8_CHUNK_SUMMARY_MAX_WORDS = 150
MIN_COVERAGE_SCORE = 0.62
STEP8_TERM_REPAIR_MAX_ROUNDS = 2
STEP8_COVERAGE_REPAIR_PASSES = 1

# Step 9 candidate selection default.
AUTO_SELECT_BEST = False

# Backward compatibility aliases (legacy notebook variable names).
STEP5C_FALLBACK_MODELS = STEP8_FALLBACK_MODELS
STEP5C_OUTER_MAX_ATTEMPTS = STEP8_OUTER_MAX_ATTEMPTS
STEP5C_PER_ATTEMPT_TIMEOUT = STEP8_PER_ATTEMPT_TIMEOUT
STEP5C_MAX_PAYLOAD_CHARS = STEP8_MAX_PAYLOAD_CHARS

if UI_MODE == 'simple':
    print('Simple mode: you can skip Step 7 (optional).')
else:
    print('Advanced mode: run Step 7 next.')

print('Style mode:', STYLE_MODE, '-> PIPELINE_MODE:', PIPELINE_MODE)
print('Context scope:', CONTEXT_SCOPE)
if MUST_INCLUDE_TERMS.strip():
    print('Must-include terms:', MUST_INCLUDE_TERMS)

if TASK_NAME == 'plot' and INPUT_MODE == 'drive_pdf_auto':
    print("For TASK_NAME='plot', use INPUT_MODE='paste_plot_data_manual' or 'drive_file_manual'.")

print('Core settings captured. Step 8 is required next; run Step 7 first only if you want optional advanced tuning.')


## Step 7 (Optional): Advanced Settings Reference

Run **Step 7** only if you want deeper control.

Step 8 is required and separate from advanced settings. If you skip Step 7, run Step 8 next.

### Advanced Settings (this step)

| Parameter | What it controls |
|---|---|
| `AUTO_EXTRACT_MAX_ATTEMPTS` | Internal retries for extraction helper calls. |
| `STEP8_OUTER_MAX_ATTEMPTS` | Outer resilient retry loop attempts in Step 8. |
| `STEP8_PER_ATTEMPT_TIMEOUT` | Timeout per Step 8 API attempt (seconds). |
| `STEP8_MAX_PAYLOAD_CHARS` | Legacy cap for payload-sized operations (kept for compatibility). |
| `CHUNK_SIZE_CHARS` | Chunk size for full-paper/method extraction coverage pipeline. |
| `CHUNK_OVERLAP_CHARS` | Overlap between consecutive chunks to reduce boundary loss. |
| `STEP8_CHUNK_SUMMARY_MAX_WORDS` | Per-chunk summary length budget before merge. |
| `MIN_COVERAGE_SCORE` | Minimum Step 8 coverage score before accepting extraction output. |
| `STEP8_TERM_REPAIR_MAX_ROUNDS` | Max rounds for term-lock repair when MUST_INCLUDE_TERMS is set. |
| `STEP8_COVERAGE_REPAIR_PASSES` | Extra repair passes if coverage score is low or terms are missing. |
| `STEP8_FALLBACK_MODELS` | Comma-separated fallback text models if primary model fails. |
| `ASPECT_RATIO` | Figure layout ratio (`21:9`, `16:9`, `3:2`). |
| `PIPELINE_MODE` | Pipeline strategy. `auto_from_style` follows Step 6 `STYLE_MODE`; or pick a pipeline explicitly. |
| `RETRIEVAL_SETTING` | Use reference retrieval (`none/auto/random/manual`). |
| `MAX_CRITIC_ROUNDS` | Number of iterative critic refinement rounds. |
| `NUM_CANDIDATES` | Number of candidates in advanced mode. |
| `MAX_CONCURRENT` | Parallel workers for advanced generation. |
| `SELECTED_CANDIDATE_INDEX` | Manual fallback index when auto-selection is disabled/fails. |
| `AUTO_SELECT_BEST` | Auto-score candidates by term coverage + caption alignment and pick best. |
| `RUN_REFINEMENT` | Enables extra post-generation refinement call. |
| `REFINE_SOURCE` | Refine selected output or an external image path. |
| `EXTERNAL_REFINE_IMAGE_PATH` | External file path used when `REFINE_SOURCE='external_image_path'`. |
| `REFINE_PROMPT` | Instruction text for refinement step. |
| `REFINE_RESOLUTION` | Refinement output size (`2K` or `4K`). |
| `REFINE_ASPECT_RATIO` | Aspect ratio for refinement output. |
| `EXPORT_4K` | Saves a 4K resized output image. |
| `ZIP_EXPORT` | Exports all run artifacts as zip bundle. |
| `SHOW_STAGE_DESCRIPTIONS` | Shows stage text in Step 10 viewer. |
| `SHOW_CRITIC_SUGGESTIONS` | Shows critic feedback text in Step 10 viewer. |



In [ ]:
#@title Step 7 (optional): Advanced settings (expand only if needed)
if UI_MODE != 'advanced':
    print('UI_MODE is simple. Skipping advanced tuning is fine.')

# Step 8 retry/fallback tuning
AUTO_EXTRACT_MAX_ATTEMPTS = 3  # @param {type:"slider", min:1, max:4, step:1}
STEP8_OUTER_MAX_ATTEMPTS = 4  # @param {type:"slider", min:1, max:4, step:1}
STEP8_PER_ATTEMPT_TIMEOUT = 60  # @param {type:"slider", min:20, max:180, step:5}
STEP8_MAX_PAYLOAD_CHARS = 50000  # @param {type:"slider", min:20000, max:90000, step:2000}
STEP8_FALLBACK_MODELS = 'gemini-2.5-pro,gemini-3.1-pro-preview'  # @param {type:"string"}
# Example fallback: gemini-2.5-pro,gemini-3.1-pro-preview

# Step 8 chunk/coverage controls
CHUNK_SIZE_CHARS = 9000  # @param {type:"slider", min:4000, max:18000, step:500}
CHUNK_OVERLAP_CHARS = 1200  # @param {type:"slider", min:200, max:3000, step:100}
STEP8_CHUNK_SUMMARY_MAX_WORDS = 150  # @param {type:"slider", min:90, max:280, step:10}
MIN_COVERAGE_SCORE = 0.62  # @param {type:"number"}
STEP8_TERM_REPAIR_MAX_ROUNDS = 2  # @param {type:"slider", min:0, max:4, step:1}
STEP8_COVERAGE_REPAIR_PASSES = 1  # @param {type:"slider", min:0, max:3, step:1}

# Step 9 selection control
AUTO_SELECT_BEST = False  # @param {type:"boolean"}
SELECTION_STRICT_REQUIRED_TERMS = True  # @param {type:"boolean"}
IMAGE_AWARE_SELECTION = False  # @param {type:"boolean"}
IMAGE_AWARE_TOPK = 3  # @param {type:"slider", min:1, max:10, step:1}
IMAGE_AWARE_WEIGHT = 0.35  # @param {type:"number"}
IMAGE_AWARE_SELECTION_MODEL = ''  # @param {type:"string"}

# Backward compatibility aliases (legacy notebook variable names).
STEP5C_FALLBACK_MODELS = STEP8_FALLBACK_MODELS
STEP5C_OUTER_MAX_ATTEMPTS = STEP8_OUTER_MAX_ATTEMPTS
STEP5C_PER_ATTEMPT_TIMEOUT = STEP8_PER_ATTEMPT_TIMEOUT
STEP5C_MAX_PAYLOAD_CHARS = STEP8_MAX_PAYLOAD_CHARS

# Pipeline controls
ASPECT_RATIO = '16:9'  # @param ['21:9', '16:9', '3:2']
# auto_from_style: follows Step 6 STYLE_MODE (matches Streamlit pipeline style switch)
# demo_planner_critic: faster, good quality
# demo_full: slower, best quality (includes Stylist)
PIPELINE_MODE = 'auto_from_style'  # @param ['auto_from_style', 'vanilla', 'dev_planner', 'dev_planner_stylist', 'dev_planner_critic', 'dev_full', 'demo_planner_critic', 'demo_full']
if PIPELINE_MODE == 'auto_from_style':
    PIPELINE_MODE = 'demo_full' if STYLE_MODE == 'stylist_enhanced' else 'demo_planner_critic'
RETRIEVAL_SETTING = 'none'  # @param ['none', 'auto', 'random', 'manual']
MAX_CRITIC_ROUNDS = 3  # @param {type:"slider", min:1, max:3, step:1}

# Batch/refinement controls
NUM_CANDIDATES = 10  # @param {type:"slider", min:1, max:20, step:1}
MAX_CONCURRENT = 10  # @param {type:"slider", min:1, max:20, step:1}
SELECTED_CANDIDATE_INDEX = 0  # @param {type:"integer"}
RUN_REFINEMENT = False  # @param {type:"boolean"}
REFINE_SOURCE = 'selected_result'  # @param ['selected_result', 'external_image_path']
EXTERNAL_REFINE_IMAGE_PATH = ''  # @param {type:"string"}
REFINE_PROMPT = 'Keep everything the same but output a cleaner higher-resolution publication-quality figure.'  # @param {type:"string"}
REFINE_RESOLUTION = '2K'  # @param ['2K', '4K']
REFINE_ASPECT_RATIO = '16:9'  # @param ['21:9', '16:9', '3:2']
EXPORT_4K = True  # @param {type:"boolean"}
ZIP_EXPORT = True  # @param {type:"boolean"}
SHOW_STAGE_DESCRIPTIONS = True  # @param {type:"boolean"}
SHOW_CRITIC_SUGGESTIONS = True  # @param {type:"boolean"}

print('Advanced settings captured. Effective pipeline mode:', PIPELINE_MODE)
print('Context scope:', CONTEXT_SCOPE)
print('Auto-select best candidate:', bool(AUTO_SELECT_BEST))
print('Strict required-term gate:', bool(SELECTION_STRICT_REQUIRED_TERMS))
print('Image-aware rerank:', bool(IMAGE_AWARE_SELECTION))
print('Next run Step 8.')



In [ ]:
#@title Step 8: Prepare source context, visual intent, and caption
print('Step 8 is required. Run this even if you skipped Step 7.')

import asyncio
import importlib
import json
import os
import random
import re
from pathlib import Path

from pypdf import PdfReader
from google.genai import types
import json_repair

import utils.generation_utils as generation_utils
importlib.reload(generation_utils)

# Concurrency guard: only one Step 8 extraction call in this runtime.
if 'STEP8_EXTRACTION_LOCK' not in globals():
    globals()['STEP8_EXTRACTION_LOCK'] = globals().get('STEP5C_EXTRACTION_LOCK', asyncio.Lock())
STEP8_EXTRACTION_LOCK = globals()['STEP8_EXTRACTION_LOCK']
# Keep legacy alias for older cells that still reference STEP5C_* names.
globals()['STEP5C_EXTRACTION_LOCK'] = STEP8_EXTRACTION_LOCK

if STEP8_EXTRACTION_LOCK.locked():
    print('Warning: Step 8 extraction is already running. Wait for it to finish or restart the runtime.')


SECTION_KEYWORD_GROUPS = {
    'problem_goal': ['problem', 'goal', 'objective', 'aim', 'hypothesis', 'challenge'],
    'pipeline_flow': ['pipeline', 'workflow', 'step', 'first', 'then', 'finally', 'process'],
    'modules': ['module', 'component', 'architecture', 'model', 'encoder', 'decoder', 'agent', 'branch'],
    'data_analysis': ['dataset', 'experiment', 'analysis', 'evaluation', 'metric', 'ablation', 'result'],
    'tools_outputs': ['tool', 'software', 'assay', 'measurement', 'output', 'figure', 'plot'],
}


def parse_page_range(range_text: str, total_pages: int):
    text = (range_text or '').strip()
    if not text:
        return list(range(total_pages))
    if '-' in text:
        left, right = text.split('-', 1)
        start = max(1, int(left.strip()))
        end = min(total_pages, int(right.strip()))
        if end < start:
            raise ValueError(f'Invalid PDF_PAGE_RANGE: {range_text}')
        return list(range(start - 1, end))
    page_num = int(text)
    if page_num < 1 or page_num > total_pages:
        raise ValueError(f'PDF page must be in [1, {total_pages}]')
    return [page_num - 1]


def resolve_input_path(default_path: str) -> str:
    chosen_path = default_path

    selected_drive = str(globals().get('SELECTED_DRIVE_FILE', '') or '').strip()
    if USE_WIDGET_SELECTION and selected_drive and Path(selected_drive).exists():
        return selected_drive

    if USE_WIDGET_SELECTION and 'pdf_chooser' in globals():
        selected = (getattr(pdf_chooser, 'selected', None) or '').strip()
        if selected and Path(selected).exists():
            return selected

    fallback = str(globals().get('FALLBACK_SELECTED_PATH', '') or '').strip()
    if fallback and Path(fallback).exists():
        return fallback

    return chosen_path


def load_text_file(path_str: str, pdf_page_range: str) -> str:
    path = Path(path_str).expanduser()
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')

    suffix = path.suffix.lower()
    if suffix in {'.txt', '.md', '.tex', '.json', '.csv', '.tsv'}:
        return path.read_text(encoding='utf-8', errors='ignore')

    if suffix == '.pdf':
        reader = PdfReader(str(path))
        page_indexes = parse_page_range(pdf_page_range, len(reader.pages))
        extracted = []
        for idx in page_indexes:
            extracted.append(reader.pages[idx].extract_text() or '')
        return '\n\n'.join(extracted)

    raise ValueError('Unsupported file type. Use .pdf, .txt, .md, .tex, .json, .csv, or .tsv.')


def build_method_candidate_text(full_text: str) -> str:
    text = re.sub(r'\n{3,}', '\n\n', full_text)
    lines = text.splitlines()

    heading_idx = []
    pattern = re.compile(r'(^|\b)(\d+\.?\s*)?(method|methodology|approach|framework|materials and methods|proposed|our model|architecture|system design|technical)(\b|$)', re.I)

    for i, line in enumerate(lines):
        s = line.strip()
        if not s:
            continue
        if len(s) <= 120 and pattern.search(s):
            heading_idx.append(i)

    if not heading_idx:
        return text[:180000]

    chunks = []
    for start in heading_idx[:3]:
        end = min(len(lines), start + 520)
        chunks.append('\n'.join(lines[start:end]))

    candidate = '\n\n'.join(chunks)
    return candidate[:180000]


def split_text_chunks(text: str, chunk_size: int, overlap: int):
    body = (text or '').strip()
    if not body:
        return []

    chunk_size = max(1200, int(chunk_size))
    overlap = max(0, int(overlap))
    overlap = min(overlap, chunk_size // 2)

    chunks = []
    start = 0
    n = len(body)

    while start < n:
        target_end = min(n, start + chunk_size)
        end = target_end

        if end < n:
            right_window = body[end:min(n, end + 180)]
            m = re.search(r'\s', right_window)
            if m:
                end = end + m.start()
            else:
                left_window = body[max(start + 1, end - 180):end]
                m2 = re.search(r'\s[^\s]*$', left_window)
                if m2:
                    end = max(start + 1, max(start + 1, end - 180) + m2.start())

        part = body[start:end].strip()
        if part:
            chunks.append(part)

        if end >= n:
            break
        start = max(0, end - overlap)

    return chunks


def parse_required_terms(raw_text: str):
    raw = str(raw_text or '').strip()
    if not raw:
        return []
    terms = []
    seen = set()
    for item in re.split(r'[\n,;]+', raw):
        t = re.sub(r'\s+', ' ', item).strip()
        if not t:
            continue
        key = t.lower()
        if key in seen:
            continue
        terms.append(t)
        seen.add(key)
    return terms


def parse_fallback_models(primary_model: str):
    raw = (
        globals().get('STEP8_FALLBACK_MODELS')
        or globals().get('STEP5C_FALLBACK_MODELS')
        or os.getenv('STEP8_FALLBACK_MODELS', '')
        or os.getenv('STEP5C_FALLBACK_MODELS', '')
    )

    if isinstance(raw, str):
        models = [x.strip() for x in raw.split(',') if x.strip()]
    elif isinstance(raw, (list, tuple, set)):
        models = [str(x).strip() for x in raw if str(x).strip()]
    else:
        models = []

    if not models:
        auto_candidates = []
        text_model_map = globals().get('TEXT_MODEL_MAP', {})
        for key in ['balanced_2_5_pro', 'fast_2_5_flash', 'paper_style_updated']:
            m = str(text_model_map.get(key, '')).strip()
            if m:
                auto_candidates.append(m)

        validation_model = str(globals().get('VALIDATION_MODEL', '')).strip()
        if validation_model:
            auto_candidates.append(validation_model)

        auto_candidates.append('gemini-2.5-flash')
        models = auto_candidates

    deduped = []
    seen = {str(primary_model).strip()}
    for m in models:
        m = str(m).strip()
        if m and m not in seen:
            deduped.append(m)
            seen.add(m)
    return deduped


def extract_error_info(error_text: str) -> dict:
    text = str(error_text or '').strip()
    upper = text.upper()

    status_match = re.search(r"'status'\s*:\s*'([^']+)'", text)
    reason_match = re.search(r"'reason'\s*:\s*'([^']+)'", text)
    code_match = re.search(r"'code'\s*:\s*(\d+)", text)

    status = status_match.group(1) if status_match else ''
    reason = reason_match.group(1) if reason_match else ''
    code = code_match.group(1) if code_match else ''

    transient_markers = [
        '503', '429', 'UNAVAILABLE', 'RESOURCE_EXHAUSTED', 'DEADLINE_EXCEEDED',
        'TIMEOUT', 'TIMED OUT', 'CONNECTION RESET', 'CONNECTION ABORTED',
    ]
    non_transient_markers = [
        'API_KEY_INVALID', 'INVALID_ARGUMENT', 'UNAUTHENTICATED', 'PERMISSION_DENIED',
        'MALFORMED', 'BAD_REQUEST', 'NOT_FOUND', 'MODEL_NOT_FOUND',
    ]

    is_transient = any(marker in upper for marker in transient_markers)
    is_non_transient = any(marker in upper for marker in non_transient_markers)

    return {
        'code': code,
        'status': status,
        'reason': reason,
        'is_transient': bool(is_transient and not is_non_transient),
        'is_non_transient': bool(is_non_transient),
        'raw': text,
    }


async def diagnose_gemini_error(model_name: str) -> str:
    client = getattr(generation_utils, 'gemini_client', None)
    if client is None:
        return 'Gemini client missing (likely GOOGLE_API_KEY not set in runtime).'

    try:
        await client.aio.models.generate_content(
            model=model_name,
            contents=[types.Part.from_text(text='ping')],
            config=types.GenerateContentConfig(
                temperature=0,
                candidate_count=1,
                max_output_tokens=4,
            ),
        )
        return ''
    except Exception as e:
        return str(e)


async def call_step8_resilient(
    prompt,
    primary_model,
    fallback_models,
    max_attempts,
    base_delay,
    max_delay,
    per_attempt_timeout,
):
    if not str(primary_model or '').strip():
        raise ValueError('STEP8: primary model is empty.')

    if not str(os.environ.get('GOOGLE_API_KEY', '')).strip():
        raise ValueError('STEP8: GOOGLE_API_KEY is missing. Re-run Step 4 and set API key.')

    model_order = [primary_model] + [m for m in (fallback_models or []) if m and m != primary_model]
    diagnostics_done = set()
    failure_log = []

    async def _single_call(model_name: str, prompt_text: str):
        response_list = await generation_utils.call_gemini_with_retry_async(
            model_name=model_name,
            contents=[{'type': 'text', 'text': prompt_text}],
            config=types.GenerateContentConfig(
                temperature=0.2,
                candidate_count=1,
                max_output_tokens=8192,
            ),
            max_attempts=1,
            retry_delay=1,
            error_context='Step8',
        )

        if (not response_list) or (not response_list[0]) or (response_list[0].strip() == 'Error'):
            raise RuntimeError('GENERATION_UTILS_ERROR')
        return (response_list[0] or '').strip()

    for model_idx, model_name in enumerate(model_order):
        for attempt in range(1, int(max_attempts) + 1):
            try:
                result_text = await asyncio.wait_for(
                    _single_call(model_name, prompt),
                    timeout=float(per_attempt_timeout),
                )
                print(f'[Step8] model={model_name} attempt={attempt}/{max_attempts} -> success')
                globals()['STEP8_LAST_SUCCESS_MODEL'] = model_name
                globals()['STEP5C_LAST_SUCCESS_MODEL'] = model_name
                return result_text

            except asyncio.TimeoutError as e:
                info = extract_error_info('TIMEOUT')
                is_transient = True
                raw_error = str(e)

            except Exception as e:
                raw_error = str(e)
                info = extract_error_info(raw_error)

                if raw_error == 'GENERATION_UTILS_ERROR' and model_name not in diagnostics_done:
                    diagnostics_done.add(model_name)
                    diag_text = await diagnose_gemini_error(model_name)
                    if diag_text:
                        info = extract_error_info(diag_text)
                        raw_error = f'{raw_error} | diagnostic={diag_text}'

                if info['is_non_transient']:
                    combined = f"{info.get('status', '')} {info.get('reason', '')} {raw_error}".upper()
                    auth_markers = ['API_KEY_INVALID', 'UNAUTHENTICATED', 'PERMISSION_DENIED']
                    is_auth_error = any(marker in combined for marker in auth_markers)

                    if is_auth_error:
                        print(
                            f"[Step8] model={model_name} attempt={attempt}/{max_attempts} "
                            f"code={info['code'] or '?'} status={info['status'] or info['reason'] or 'auth_error'} -> fail-fast"
                        )
                        raise ValueError(
                            'Step 8 extraction authentication/permission error. '
                            f"model={model_name}, code={info['code'] or '?'} status={info['status'] or info['reason'] or 'unknown'}. "
                            'Re-run Step 4 with a valid API key.'
                        ) from e

                    if model_idx < len(model_order) - 1:
                        print(
                            f"[Step8] model={model_name} attempt={attempt}/{max_attempts} "
                            f"code={info['code'] or '?'} status={info['status'] or info['reason'] or 'non_transient'} -> trying next fallback model"
                        )
                        break

                    print(
                        f"[Step8] model={model_name} attempt={attempt}/{max_attempts} "
                        f"code={info['code'] or '?'} status={info['status'] or info['reason'] or 'non_transient'} -> fail-fast"
                    )
                    raise ValueError(
                        'Step 8 extraction non-transient API error. '
                        f"model={model_name}, code={info['code'] or '?'} status={info['status'] or info['reason'] or 'unknown'}. "
                        f'Details: {raw_error}'
                    ) from e

                is_transient = info['is_transient'] or (raw_error == 'GENERATION_UTILS_ERROR')

            failure_log.append({
                'model': model_name,
                'attempt': attempt,
                'code': info.get('code', ''),
                'status': info.get('status', ''),
                'reason': info.get('reason', ''),
                'raw': raw_error,
            })

            if not is_transient:
                print(
                    f"[Step8] model={model_name} attempt={attempt}/{max_attempts} "
                    f"code={info.get('code') or '?'} status={info.get('status') or info.get('reason') or 'unknown'} -> fail-fast"
                )
                raise ValueError(
                    'Step 8 extraction failed with a non-transient error. '
                    f"model={model_name}, code={info.get('code') or '?'}, "
                    f"status={info.get('status') or info.get('reason') or 'unknown'}. Details: {raw_error}"
                )

            at_model_end = attempt >= int(max_attempts)
            has_next_model = model_idx < (len(model_order) - 1)

            if at_model_end and has_next_model:
                print(
                    f"[Step8] model={model_name} exhausted after {max_attempts} transient attempts. "
                    f"Switching to fallback model: {model_order[model_idx + 1]}"
                )
                break

            if at_model_end and not has_next_model:
                break

            delay = min(float(base_delay) * (2 ** (attempt - 1)), float(max_delay))
            jitter = random.uniform(0, max(0.001, delay * 0.2))
            wait_s = delay + jitter

            print(
                f"[Step8] model={model_name} attempt={attempt}/{max_attempts} "
                f"code={info.get('code') or '?'} status={info.get('status') or info.get('reason') or 'transient'} "
                f"-> retry in {wait_s:.1f}s"
            )
            await asyncio.sleep(wait_s)

    if failure_log:
        last = failure_log[-1]
        raise RuntimeError(
            'Step 8 extraction: all retry attempts failed across primary/fallback models. '
            f"Last failure model={last['model']} code={last['code'] or '?'} "
            f"status={last['status'] or last['reason'] or 'unknown'}. "
            'Try: re-run Step 4 with fast_2_5_flash, set STEP8_FALLBACK_MODELS=gemini-2.5-pro,gemini-3.1-pro-preview, reduce PDF_PAGE_RANGE, or switch INPUT_MODE to paste_text_manual.'
        )

    raise RuntimeError('Step 8 extraction: retry wrapper ended without response.')


def parse_extraction_json(raw_text: str) -> dict:
    cleaned = (raw_text or '').replace('```json', '').replace('```', '').strip()
    obj = json_repair.loads(cleaned)
    if not isinstance(obj, dict):
        raise ValueError('Model output is not a JSON dictionary.')
    return {
        'source_context': str(obj.get('source_context', '')).strip(),
        'visual_intent': str(obj.get('visual_intent', '')).strip(),
        'figure_caption': str(obj.get('figure_caption', '')).strip(),
    }


def validate_extraction_fields(data: dict):
    missing = [k for k in ['source_context', 'visual_intent', 'figure_caption'] if not str(data.get(k, '')).strip()]
    if missing:
        raise ValueError(f'Missing required keys after extraction: {missing}')


def compute_coverage_report(parsed: dict, required_terms: list[str]) -> dict:
    source = str(parsed.get('source_context', '')).strip()
    visual = str(parsed.get('visual_intent', '')).strip()
    caption = str(parsed.get('figure_caption', '')).strip()

    combined = f'{source}\n{visual}\n{caption}'.lower()

    section_hits = {}
    for section_name, keywords in SECTION_KEYWORD_GROUPS.items():
        section_hits[section_name] = any(k.lower() in combined for k in keywords)

    section_score = sum(1.0 for hit in section_hits.values() if hit) / max(1, len(section_hits))

    missing_source_terms = []
    missing_visual_caption_terms = []
    term_hits = {}
    source_lower = source.lower()
    vis_cap_lower = f'{visual}\n{caption}'.lower()
    combined_lower = combined

    for term in required_terms:
        t = term.lower().strip()
        in_source = t in source_lower
        in_vis_cap = t in vis_cap_lower
        in_combined = t in combined_lower
        term_hits[term] = bool(in_combined)
        if not in_source:
            missing_source_terms.append(term)
        if not in_vis_cap:
            missing_visual_caption_terms.append(term)

    term_score = 1.0 if not required_terms else (sum(1.0 for t in required_terms if term_hits.get(t, False)) / len(required_terms))
    total_score = 0.75 * section_score + 0.25 * term_score

    missing_sections = [k for k, hit in section_hits.items() if not hit]

    return {
        'score': float(total_score),
        'section_score': float(section_score),
        'term_score': float(term_score),
        'section_hits': section_hits,
        'missing_sections': missing_sections,
        'missing_source_terms': missing_source_terms,
        'missing_visual_caption_terms': missing_visual_caption_terms,
        'missing_terms': [t for t in required_terms if not term_hits.get(t, False)],
    }


async def summarize_chunks(chunk_texts, intent_hint, primary_model, fallback_models, max_attempts, base_delay, max_delay, per_attempt_timeout, max_words):
    briefs = []
    total = len(chunk_texts)
    for idx, chunk_text in enumerate(chunk_texts, start=1):
        prompt = f"""
You are extracting figure-relevant facts from a paper chunk.

Return plain text only under {int(max_words)} words with these labels exactly:
- Problem/Goal:
- Method Flow:
- Modules/Components:
- Data/Analysis:
- Tools/Entities:

Prioritize concrete nouns, entities, measurements, and explicit procedure steps.
Do not add details not present in the chunk.

User hint:
{intent_hint}

Paper chunk {idx}/{total}:
<BEGIN_CHUNK>
{chunk_text}
<END_CHUNK>
"""
        raw = await call_step8_resilient(
            prompt=prompt,
            primary_model=primary_model,
            fallback_models=fallback_models,
            max_attempts=max_attempts,
            base_delay=base_delay,
            max_delay=max_delay,
            per_attempt_timeout=per_attempt_timeout,
        )
        brief = (raw or '').replace('```', '').strip()
        briefs.append(brief)
        print(f'[Step8] chunk summary {idx}/{total} done ({len(brief)} chars).')
    return briefs


async def build_structured_inputs(
    merged_summary_text,
    intent_hint,
    required_terms,
    primary_model,
    fallback_models,
    max_attempts,
    base_delay,
    max_delay,
    per_attempt_timeout,
):
    required_terms_text = ', '.join(required_terms) if required_terms else '(none)'

    pass_b_prompt = f"""
You are assisting PaperBanana setup for one publication-quality method figure.

From the merged chunk summaries, return STRICT JSON only with exactly keys:
- source_context
- visual_intent
- figure_caption

Hard requirements:
- source_context: 1200-2800 words, faithful, specific, and section-aware.
- source_context must include: problem/goal, pipeline flow, modules/components, data/analysis details, tools/entities.
- visual_intent: one concise sentence focused on what to draw.
- figure_caption: 1-2 sentences starting with 'Figure 1:'.
- Avoid generic phrases. Prefer paper-specific terms.
- Do not include markdown fences or extra keys.

Must-include terms (if provided):
{required_terms_text}

User hint:
{intent_hint}

Merged chunk summaries:
<BEGIN_SUMMARY>
{merged_summary_text}
<END_SUMMARY>
"""

    pass_b_raw = await call_step8_resilient(
        prompt=pass_b_prompt,
        primary_model=primary_model,
        fallback_models=fallback_models,
        max_attempts=max_attempts,
        base_delay=base_delay,
        max_delay=max_delay,
        per_attempt_timeout=per_attempt_timeout,
    )

    try:
        parsed = parse_extraction_json(pass_b_raw)
        validate_extraction_fields(parsed)
    except Exception:
        repair_prompt = f"""
Return STRICT JSON only (no markdown, no explanation), with exactly keys:
- source_context
- visual_intent
- figure_caption

Repair this output into valid JSON if malformed:
<BEGIN_PREVIOUS_OUTPUT>
{pass_b_raw}
<END_PREVIOUS_OUTPUT>

Use this merged summary as ground truth:
<BEGIN_SUMMARY>
{merged_summary_text}
<END_SUMMARY>
"""
        repaired_raw = await call_step8_resilient(
            prompt=repair_prompt,
            primary_model=primary_model,
            fallback_models=fallback_models,
            max_attempts=max(1, int(max_attempts) - 1),
            base_delay=base_delay,
            max_delay=max_delay,
            per_attempt_timeout=per_attempt_timeout,
        )
        parsed = parse_extraction_json(repaired_raw)
        validate_extraction_fields(parsed)

    return parsed


async def repair_for_term_lock(
    parsed,
    merged_summary_text,
    required_terms,
    coverage,
    primary_model,
    fallback_models,
    max_attempts,
    base_delay,
    max_delay,
    per_attempt_timeout,
    max_rounds,
):
    if not required_terms:
        return parsed

    out = dict(parsed)
    for round_idx in range(1, max(0, int(max_rounds)) + 1):
        coverage = compute_coverage_report(out, required_terms)
        if not coverage['missing_source_terms'] and not coverage['missing_visual_caption_terms']:
            return out

        repair_prompt = f"""
Return STRICT JSON only with exactly keys:
- source_context
- visual_intent
- figure_caption

Revise the JSON so term-lock constraints pass.

Missing in source_context:
{coverage['missing_source_terms']}

Missing in visual_intent/figure_caption:
{coverage['missing_visual_caption_terms']}

Keep the same factual meaning and rely on this summary only:
<BEGIN_SUMMARY>
{merged_summary_text}
<END_SUMMARY>

Current JSON:
<BEGIN_JSON>
{json.dumps(out, ensure_ascii=False)}
<END_JSON>
"""
        repaired_raw = await call_step8_resilient(
            prompt=repair_prompt,
            primary_model=primary_model,
            fallback_models=fallback_models,
            max_attempts=max(1, int(max_attempts) - 1),
            base_delay=base_delay,
            max_delay=max_delay,
            per_attempt_timeout=per_attempt_timeout,
        )
        out = parse_extraction_json(repaired_raw)
        validate_extraction_fields(out)
        print(f'[Step8] term-lock repair round {round_idx} completed.')

    return out


async def repair_for_coverage(
    parsed,
    merged_summary_text,
    required_terms,
    coverage,
    intent_hint,
    primary_model,
    fallback_models,
    max_attempts,
    base_delay,
    max_delay,
    per_attempt_timeout,
):
    missing_sections = coverage.get('missing_sections', [])
    missing_terms = coverage.get('missing_terms', [])

    repair_prompt = f"""
Return STRICT JSON only with keys:
- source_context
- visual_intent
- figure_caption

Coverage repair request:
- Missing sections: {missing_sections}
- Missing terms: {missing_terms}

Requirements:
- Keep faithful to source summaries.
- Increase specificity for missing sections.
- Keep visual_intent as one sentence.
- Keep figure_caption 1-2 sentences and start with 'Figure 1:'.

User hint:
{intent_hint}

Merged summaries:
<BEGIN_SUMMARY>
{merged_summary_text}
<END_SUMMARY>

Current JSON:
<BEGIN_JSON>
{json.dumps(parsed, ensure_ascii=False)}
<END_JSON>
"""

    repaired_raw = await call_step8_resilient(
        prompt=repair_prompt,
        primary_model=primary_model,
        fallback_models=fallback_models,
        max_attempts=max(1, int(max_attempts) - 1),
        base_delay=base_delay,
        max_delay=max_delay,
        per_attempt_timeout=per_attempt_timeout,
    )

    out = parse_extraction_json(repaired_raw)
    validate_extraction_fields(out)
    return out


async def run_scope_extraction(
    scope_name,
    source_text,
    intent_hint,
    required_terms,
    settings,
):
    chunk_texts = split_text_chunks(
        source_text,
        int(settings['chunk_size']),
        int(settings['chunk_overlap']),
    )
    if not chunk_texts:
        raise ValueError(f'No extractable chunks found for scope={scope_name}.')

    print(f"[Step8] scope={scope_name} | chars={len(source_text)} | chunks={len(chunk_texts)}")

    chunk_briefs = await summarize_chunks(
        chunk_texts,
        intent_hint,
        settings['primary_model'],
        settings['fallback_models'],
        settings['outer_attempts'],
        settings['base_delay'],
        settings['max_delay'],
        settings['per_attempt_timeout'],
        settings['chunk_summary_max_words'],
    )

    merged_summary = []
    for i, brief in enumerate(chunk_briefs, start=1):
        merged_summary.append(f'[Chunk {i}/{len(chunk_briefs)}]\n{brief}')
    merged_summary_text = '\n\n'.join(merged_summary)

    parsed = await build_structured_inputs(
        merged_summary_text=merged_summary_text,
        intent_hint=intent_hint,
        required_terms=required_terms,
        primary_model=settings['primary_model'],
        fallback_models=settings['fallback_models'],
        max_attempts=settings['outer_attempts'],
        base_delay=settings['base_delay'],
        max_delay=settings['max_delay'],
        per_attempt_timeout=settings['per_attempt_timeout'],
    )

    parsed = await repair_for_term_lock(
        parsed=parsed,
        merged_summary_text=merged_summary_text,
        required_terms=required_terms,
        coverage=compute_coverage_report(parsed, required_terms),
        primary_model=settings['primary_model'],
        fallback_models=settings['fallback_models'],
        max_attempts=settings['outer_attempts'],
        base_delay=settings['base_delay'],
        max_delay=settings['max_delay'],
        per_attempt_timeout=settings['per_attempt_timeout'],
        max_rounds=settings['term_repair_rounds'],
    )

    coverage = compute_coverage_report(parsed, required_terms)

    for repair_idx in range(1, max(0, int(settings['coverage_repair_passes'])) + 1):
        if (coverage['score'] >= settings['min_coverage_score']) and not coverage['missing_terms']:
            break
        print(
            f"[Step8] coverage repair pass {repair_idx}: score={coverage['score']:.3f}, "
            f"missing_sections={coverage['missing_sections']}, missing_terms={coverage['missing_terms']}"
        )
        parsed = await repair_for_coverage(
            parsed=parsed,
            merged_summary_text=merged_summary_text,
            required_terms=required_terms,
            coverage=coverage,
            intent_hint=intent_hint,
            primary_model=settings['primary_model'],
            fallback_models=settings['fallback_models'],
            max_attempts=settings['outer_attempts'],
            base_delay=settings['base_delay'],
            max_delay=settings['max_delay'],
            per_attempt_timeout=settings['per_attempt_timeout'],
        )
        parsed = await repair_for_term_lock(
            parsed=parsed,
            merged_summary_text=merged_summary_text,
            required_terms=required_terms,
            coverage=compute_coverage_report(parsed, required_terms),
            primary_model=settings['primary_model'],
            fallback_models=settings['fallback_models'],
            max_attempts=settings['outer_attempts'],
            base_delay=settings['base_delay'],
            max_delay=settings['max_delay'],
            per_attempt_timeout=settings['per_attempt_timeout'],
            max_rounds=max(0, settings['term_repair_rounds'] - 1),
        )
        coverage = compute_coverage_report(parsed, required_terms)

    return parsed, coverage, merged_summary_text, len(chunk_texts)


async def auto_extract_inputs_from_paper(full_text: str, intent_hint: str, max_attempts: int, context_scope: str, required_terms: list[str]):
    async with STEP8_EXTRACTION_LOCK:
        settings = {
            'primary_model': MODEL_NAME,
            'fallback_models': parse_fallback_models(MODEL_NAME),
            'outer_attempts': int(globals().get('STEP8_OUTER_MAX_ATTEMPTS', globals().get('STEP5C_OUTER_MAX_ATTEMPTS', max(1, int(max_attempts))))),
            'base_delay': float(globals().get('STEP8_BASE_DELAY', globals().get('STEP5C_BASE_DELAY', 2.0))),
            'max_delay': float(globals().get('STEP8_MAX_DELAY', globals().get('STEP5C_MAX_DELAY', 20.0))),
            'per_attempt_timeout': float(globals().get('STEP8_PER_ATTEMPT_TIMEOUT', globals().get('STEP5C_PER_ATTEMPT_TIMEOUT', 60.0))),
            'chunk_size': int(globals().get('CHUNK_SIZE_CHARS', 9000)),
            'chunk_overlap': int(globals().get('CHUNK_OVERLAP_CHARS', 1200)),
            'chunk_summary_max_words': int(globals().get('STEP8_CHUNK_SUMMARY_MAX_WORDS', 150)),
            'min_coverage_score': float(globals().get('MIN_COVERAGE_SCORE', 0.62)),
            'term_repair_rounds': int(globals().get('STEP8_TERM_REPAIR_MAX_ROUNDS', 2)),
            'coverage_repair_passes': int(globals().get('STEP8_COVERAGE_REPAIR_PASSES', 1)),
        }

        print('[Step8] primary model:', settings['primary_model'])
        print('[Step8] fallback models:', settings['fallback_models'] if settings['fallback_models'] else 'none')
        print(
            f"[Step8] chunk settings: size={settings['chunk_size']}, overlap={settings['chunk_overlap']}, "
            f"max_words={settings['chunk_summary_max_words']}"
        )

        normalized_scope = str(context_scope or 'auto_method_first').strip().lower()
        if normalized_scope not in {'auto_method_first', 'method_only', 'full_paper'}:
            normalized_scope = 'auto_method_first'

        if (normalized_scope == 'method_only') and (not required_terms):
            print(
                "[Step8] Note: CONTEXT_SCOPE='method_only' with empty MUST_INCLUDE_TERMS may miss specific concepts "
                "(e.g., control-vs-treatment branch logic). Set CONTEXT_SCOPE='full_paper' and define MUST_INCLUDE_TERMS."
            )

        if normalized_scope == 'method_only':
            scope_order = ['method_only']
        elif normalized_scope == 'full_paper':
            scope_order = ['full_paper']
        else:
            scope_order = ['method_only', 'full_paper']

        best_candidate = None

        for scope_name in scope_order:
            if scope_name == 'method_only':
                source_text = build_method_candidate_text(full_text)
            else:
                source_text = full_text

            parsed, coverage, merged_summary_text, chunk_count = await run_scope_extraction(
                scope_name=scope_name,
                source_text=source_text,
                intent_hint=intent_hint,
                required_terms=required_terms,
                settings=settings,
            )

            candidate = {
                'parsed': parsed,
                'coverage': coverage,
                'scope': scope_name,
                'chunk_count': chunk_count,
                'source_text_chars': len(source_text),
                'merged_summary_chars': len(merged_summary_text),
                'candidate_text': source_text,
            }

            if (best_candidate is None) or (coverage['score'] > best_candidate['coverage']['score']):
                best_candidate = candidate

            passed = (coverage['score'] >= settings['min_coverage_score']) and (not coverage['missing_terms'])
            print(
                f"[Step8] scope={scope_name} coverage_score={coverage['score']:.3f} "
                f"(threshold={settings['min_coverage_score']:.2f}) missing_terms={coverage['missing_terms']}"
            )

            if passed:
                best_candidate = candidate
                break

            if normalized_scope == 'auto_method_first' and scope_name == 'method_only':
                print('[Step8] method_only coverage did not pass threshold; switching to full_paper scope.')

        if best_candidate is None:
            raise RuntimeError('Step 8 could not produce any candidate extraction output.')

        parsed = dict(best_candidate['parsed'])
        parsed['_scope_used'] = best_candidate['scope']
        parsed['_coverage_score'] = float(best_candidate['coverage']['score'])
        parsed['_coverage'] = best_candidate['coverage']
        parsed['_chunk_count'] = int(best_candidate['chunk_count'])
        parsed['_source_text_chars'] = int(best_candidate['source_text_chars'])
        parsed['_candidate_text'] = best_candidate['candidate_text']

        used_model = str(globals().get('STEP8_LAST_SUCCESS_MODEL', globals().get('STEP5C_LAST_SUCCESS_MODEL', ''))).strip()
        globals()['STEP8_FALLBACK_USED_MODEL'] = used_model
        if used_model and used_model != MODEL_NAME:
            print()
            print(
                f'WARNING: Step 8 extraction used fallback model "{used_model}" because primary "{MODEL_NAME}" failed.'
            )
            print(
                f'Step 9 will keep your original text model "{MODEL_NAME}" for predictable speed/cost.'
            )
            print('If Step 9 fails, change TEXT_MODEL_PRESET in Step 4 or set fallback models in Step 7.')
            print()

        return parsed


resolved_input_path = resolve_input_path(FALLBACK_SELECTED_PATH)
method_text = ''
visual_intent_text = ''
figure_caption_text = ''

step8_required_terms = parse_required_terms(globals().get('MUST_INCLUDE_TERMS', ''))
globals()['STEP8_REQUIRED_TERMS'] = step8_required_terms

if TASK_NAME == 'plot' and INPUT_MODE == 'drive_file_manual':
    plot_input_candidate = (PLOT_RAW_DATA_FILE_PATH.strip() or resolved_input_path).strip()
    if not Path(plot_input_candidate).exists():
        raise FileNotFoundError(
            f'Plot input file not found: {plot_input_candidate}\n'
            'Upload your file to Google Drive and update PLOT_RAW_DATA_FILE_PATH in Step 6, '
            'or switch INPUT_MODE to paste_plot_data_manual.'
        )
elif INPUT_MODE not in ('paste_text_manual', 'paste_plot_data_manual'):
    if not Path(resolved_input_path).exists():
        raise FileNotFoundError(
            f'Input file not found: {resolved_input_path}\n'
            'Upload your file to Google Drive and update FALLBACK_SELECTED_PATH in Step 5, '
            'or switch INPUT_MODE to paste_text_manual.'
        )

if TASK_NAME == 'plot':
    globals()['STEP8_CONTEXT_SCOPE_USED'] = 'plot_manual_data'
    globals()['STEP8_COVERAGE_SCORE'] = 1.0
    globals()['STEP8_MISSING_TERMS'] = []

    if INPUT_MODE == 'paste_plot_data_manual':
        method_text = PASTED_PLOT_RAW_DATA.strip()
    elif INPUT_MODE == 'drive_file_manual':
        resolved_plot_path = PLOT_RAW_DATA_FILE_PATH.strip() or resolved_input_path
        method_text = load_text_file(resolved_plot_path, '').strip()
        resolved_input_path = resolved_plot_path
    else:
        raise ValueError(
            "For TASK_NAME='plot', use INPUT_MODE='paste_plot_data_manual' or INPUT_MODE='drive_file_manual'."
        )

    visual_intent_text = (PLOT_VISUAL_INTENT or VISUAL_INTENT_HINT).strip()
    figure_caption_text = (MANUAL_FIGURE_CAPTION or f'Figure 1: {visual_intent_text}').strip()

else:
    if INPUT_MODE == 'paste_text_manual':
        globals()['STEP8_CONTEXT_SCOPE_USED'] = 'manual_paste'
        globals()['STEP8_COVERAGE_SCORE'] = 1.0
        globals()['STEP8_MISSING_TERMS'] = []

        method_text = PASTED_METHOD_TEXT.strip()
        visual_intent_text = VISUAL_INTENT_HINT.strip()
        figure_caption_text = (MANUAL_FIGURE_CAPTION or f'Figure 1: {visual_intent_text}').strip()

    elif INPUT_MODE == 'drive_file_manual':
        globals()['STEP8_CONTEXT_SCOPE_USED'] = 'manual_file'
        globals()['STEP8_COVERAGE_SCORE'] = 1.0
        globals()['STEP8_MISSING_TERMS'] = []

        method_text = load_text_file(resolved_input_path, PDF_PAGE_RANGE).strip()
        visual_intent_text = VISUAL_INTENT_HINT.strip()
        figure_caption_text = (MANUAL_FIGURE_CAPTION or f'Figure 1: {visual_intent_text}').strip()

    elif INPUT_MODE == 'drive_pdf_auto':
        if not resolved_input_path.lower().endswith('.pdf'):
            raise ValueError('drive_pdf_auto mode requires a .pdf input file.')

        full_paper_text = load_text_file(resolved_input_path, PDF_PAGE_RANGE).strip()
        context_scope = str(globals().get('CONTEXT_SCOPE', 'auto_method_first')).strip().lower()

        try:
            auto_inputs = await auto_extract_inputs_from_paper(
                full_text=full_paper_text,
                intent_hint=VISUAL_INTENT_HINT,
                max_attempts=AUTO_EXTRACT_MAX_ATTEMPTS,
                context_scope=context_scope,
                required_terms=step8_required_terms,
            )
        except Exception as e:
            raise ValueError(
                'Auto extraction failed after chunked resilient fallback policy. '
                'Check Step 4 API key, optionally set STEP8_FALLBACK_MODELS, reduce PDF_PAGE_RANGE, '
                'or switch INPUT_MODE to paste_text_manual. '
                f'Details: {e}'
            ) from e

        method_text = auto_inputs['source_context']
        visual_intent_text = auto_inputs['visual_intent'] or VISUAL_INTENT_HINT.strip()
        figure_caption_text = auto_inputs['figure_caption'] or f'Figure 1: {visual_intent_text}'

        globals()['candidate_text'] = auto_inputs.get('_candidate_text', '')
        globals()['STEP8_CONTEXT_SCOPE_USED'] = auto_inputs.get('_scope_used', context_scope)
        globals()['STEP8_COVERAGE_SCORE'] = float(auto_inputs.get('_coverage_score', 0.0))
        globals()['STEP8_MISSING_TERMS'] = list(auto_inputs.get('_coverage', {}).get('missing_terms', []))
        globals()['STEP8_CHUNK_COUNT'] = int(auto_inputs.get('_chunk_count', 0))

    else:
        raise ValueError(f'Unknown INPUT_MODE: {INPUT_MODE}')

if TASK_NAME == 'plot':
    if len(method_text) < 20:
        raise ValueError('Plot raw data is too short. Paste more data or load a valid CSV/JSON/TXT file.')
else:
    if len(method_text) < 180:
        raise ValueError('Resolved source context is too short. Try a different PDF range or use manual mode.')

if not visual_intent_text.strip():
    raise ValueError('Visual intent is empty after preparation.')

if not figure_caption_text.strip():
    figure_caption_text = f'Figure 1: {visual_intent_text.strip()}'

print('Task name:', TASK_NAME)
print('Input mode:', INPUT_MODE)
print('Resolved input path:', resolved_input_path)
print('Source content length:', len(method_text))
print('Visual intent:', visual_intent_text)
print('Figure caption:', figure_caption_text)
if TASK_NAME != 'plot':
    print('Step 8 context scope used:', globals().get('STEP8_CONTEXT_SCOPE_USED', 'n/a'))
    print('Step 8 coverage score:', f"{float(globals().get('STEP8_COVERAGE_SCORE', 0.0)):.3f}")
    if step8_required_terms:
        print('Required terms:', step8_required_terms)
        print('Missing terms after extraction:', globals().get('STEP8_MISSING_TERMS', []))
print('\nSource preview (first 900 chars):\n')
print(method_text[:900])



## Run and Output Notes

Run this after **Step 8**.

### What Step 9 does

1. Builds the pipeline with selected settings.
2. Generates image(s) from your prepared inputs.
3. Prints a cost estimate (USD) using current settings and official model pricing.
4. Displays the final image directly in the notebook.
5. Saves outputs to Drive under `PaperBanana/outputs/run_*`.

### Style modes available in notebook

- `STYLE_MODE='planner_critic_fast'` -> Streamlit-style `demo_planner_critic` pipeline.
- `STYLE_MODE='stylist_enhanced'` -> Streamlit-style `demo_full` pipeline (includes Stylist agent).


### `SAVE_FULL_RESULT_WITH_BASE64` (Step 9)

- `False` (recommended): saves smaller JSON files and keeps images as `.png/.jpg` files on Drive.
- `True`: embeds full image bytes as base64 text inside JSON (very large files; use only for debugging or self-contained exports).

### Output files you will see

- `*_raw_model_output.jpg/png`: raw image output returned by the model.
- `*_final_deliverable.png`: standardized output image (4K when `EXPORT_4K=True`).
- `full_result.json` / `all_results.json`: run metadata and text artifacts.
- `resolved_inputs.json`: exact settings used for reproducibility.
- `export_bundle.zip` (optional): all run files in one archive.

Simple mode:
- One result, fastest workflow.

Advanced mode:
- Multiple candidates are displayed in notebook and all candidate artifacts are saved under `run_*/candidates/`.
- One candidate is still marked as selected for final output/refinement.
- Optional refinement and zip export remain available.

### Step 10 (optional) viewer settings

Use Step 10 only if you want to inspect intermediate pipeline stages (Planner/Stylist/Critic).

- `STAGE_VIEW_CANDIDATE_INDEX = -1`: use the selected final result (recommended).
- `STAGE_VIEW_CANDIDATE_INDEX = 0,1,2,...`: inspect that specific advanced-mode candidate.
- `SHOW_STAGE_DESCRIPTIONS`: print text descriptions used at each stage.
- `SHOW_CRITIC_SUGGESTIONS`: print critic feedback for each round.


In [ ]:
#@title Step 9: Run generation, preview, and save outputs
import base64
import importlib
import json
import math
import re
import zipfile
from datetime import datetime
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown
from google.genai import types

# If True, JSON files will include base64 image data (much larger files).
SAVE_FULL_RESULT_WITH_BASE64 = False  # @param {type:"boolean"}

if SAVE_FULL_RESULT_WITH_BASE64:
    print('Step 9 save mode: full JSON with embedded base64 image data (large files).')
else:
    print('Step 9 save mode: slim JSON without embedded images (recommended).')

# Reload generation utils after writing config/model settings.
import utils.generation_utils as generation_utils
importlib.reload(generation_utils)

from utils import config
from utils.paperviz_processor import PaperVizProcessor
from agents.vanilla_agent import VanillaAgent
from agents.planner_agent import PlannerAgent
from agents.visualizer_agent import VisualizerAgent
from agents.stylist_agent import StylistAgent
from agents.critic_agent import CriticAgent
from agents.retriever_agent import RetrieverAgent
from agents.polish_agent import PolishAgent


def target_4k_size(aspect_ratio: str) -> tuple[int, int]:
    if aspect_ratio == '21:9':
        return (4096, 1755)
    if aspect_ratio == '3:2':
        return (3840, 2560)
    return (3840, 2160)


def pick_final_image_key(result_dict, task_name: str, max_rounds: int):
    task_name = task_name.lower().strip()
    for round_idx in range(int(max_rounds) - 1, -1, -1):
        key = f'target_{task_name}_critic_desc{round_idx}_base64_jpg'
        if result_dict.get(key):
            return key
    for key in [
        f'target_{task_name}_stylist_desc0_base64_jpg',
        f'target_{task_name}_desc0_base64_jpg',
        f'vanilla_{task_name}_base64_jpg',
    ]:
        if result_dict.get(key):
            return key
    return None


def b64_to_image(b64_str: str) -> Image.Image:
    return Image.open(BytesIO(base64.b64decode(b64_str))).convert('RGB')


def strip_base64_fields(payload):
    if isinstance(payload, dict):
        return {k: strip_base64_fields(v) for k, v in payload.items() if '_base64_' not in k}
    if isinstance(payload, list):
        return [strip_base64_fields(x) for x in payload]
    return payload


PRICING_REFERENCE_URL = 'https://ai.google.dev/gemini-api/docs/pricing'
PRICING_LAST_VERIFIED = '2026-02-27'
CHARS_PER_TOKEN_EST = 4.0

# Rates per 1M tokens (USD), verified from official Gemini pricing docs.
TEXT_PRICING_USD_PER_MTOK = {
    'gemini-2.5-flash': {'input': 0.30, 'output': 2.50},
    'gemini-2.5-pro': {'input': 1.25, 'output': 10.00},
    'gemini-3.1-pro-preview': {'input': 3.00, 'output': 15.00},
    'gemini-3.1-flash-image-preview': {'input': 0.25, 'output': 1.50},
    'gemini-3-pro-image-preview': {'input': 2.00, 'output': 12.00},
    'gemini-2.5-flash-image': {'input': 0.30, 'output': 2.50},
}

# Image output prices (USD per generated image) used by this notebook.
IMAGE_OUTPUT_USD_PER_IMAGE = {
    'gemini-3.1-flash-image-preview': {'1K': 0.067, '2K': 0.101, '4K': 0.151},
    'gemini-3-pro-image-preview': {'1K': 0.134, '2K': 0.134, '4K': 0.240},
    'gemini-2.5-flash-image': {'1K': 0.039},
}

# Image input prices for image-to-image refinement (USD per input image).
IMAGE_INPUT_USD_PER_IMAGE = {
    'gemini-3.1-flash-image-preview': 0.0006,
    'gemini-3-pro-image-preview': 0.0011,
}


def approx_tokens(text: str, chars_per_token: float = CHARS_PER_TOKEN_EST) -> int:
    t = str(text or '')
    if not t:
        return 0
    return max(1, int(math.ceil(len(t) / chars_per_token)))


def pipeline_call_ranges(pipeline_mode: str, max_critic_rounds: int, batch_size: int) -> dict:
    p = str(pipeline_mode or '').strip().lower()
    rounds = max(0, int(max_critic_rounds))
    batch = max(1, int(batch_size))

    out = {
        'planner_calls': 0,
        'stylist_calls': 0,
        'critic_calls_min': 0,
        'critic_calls_max': 0,
        'image_calls_min': 0,
        'image_calls_max': 0,
    }

    if p == 'vanilla':
        out['image_calls_min'] = batch
        out['image_calls_max'] = batch
        return out

    if p in {'dev_retriever', 'dev_polish'}:
        return out

    if p == 'dev_planner':
        out['planner_calls'] = batch
        out['image_calls_min'] = batch
        out['image_calls_max'] = batch
        return out

    if p == 'dev_planner_stylist':
        out['planner_calls'] = batch
        out['stylist_calls'] = batch
        # Visualizer currently renders planner + stylist stage images.
        out['image_calls_min'] = 2 * batch
        out['image_calls_max'] = 2 * batch
        return out

    if p in {'dev_planner_critic', 'demo_planner_critic'}:
        out['planner_calls'] = batch
        out['critic_calls_min'] = batch if rounds > 0 else 0
        out['critic_calls_max'] = batch * rounds
        out['image_calls_min'] = batch
        out['image_calls_max'] = batch + batch * rounds
        return out

    if p in {'dev_full', 'demo_full'}:
        out['planner_calls'] = batch
        out['stylist_calls'] = batch
        out['critic_calls_min'] = batch if rounds > 0 else 0
        out['critic_calls_max'] = batch * rounds
        # Visualizer currently renders planner + stylist stage images before critic rounds.
        out['image_calls_min'] = 2 * batch
        out['image_calls_max'] = (2 * batch) + (batch * rounds)
        return out

    # Conservative fallback for unknown modes.
    out['planner_calls'] = batch
    out['critic_calls_min'] = batch if rounds > 0 else 0
    out['critic_calls_max'] = batch * rounds
    out['image_calls_min'] = batch
    out['image_calls_max'] = batch + batch * rounds
    return out


def model_rates_or_none(model_name: str):
    return TEXT_PRICING_USD_PER_MTOK.get(str(model_name or '').strip())


def image_output_rate_or_none(model_name: str, resolution: str = '1K'):
    rates = IMAGE_OUTPUT_USD_PER_IMAGE.get(str(model_name or '').strip(), {})
    if resolution in rates:
        return rates[resolution], False
    if '1K' in rates:
        return rates['1K'], True
    return None, False


def count_image_keys(result_obj: dict, task_name: str) -> int:
    if not isinstance(result_obj, dict):
        return 0
    pattern = re.compile(
        rf'^(target_{task_name}_(desc0|stylist_desc0|critic_desc\d+)|vanilla_{task_name})_base64_jpg$'
    )
    return sum(1 for k, v in result_obj.items() if pattern.match(k) and isinstance(v, str) and len(v) > 20)


def parse_required_terms(raw_text: str):
    raw = str(raw_text or '').strip()
    if not raw:
        return []
    terms = []
    seen = set()
    for item in re.split(r'[\n,;]+', raw):
        t = re.sub(r'\s+', ' ', item).strip()
        if not t:
            continue
        key = t.lower()
        if key in seen:
            continue
        terms.append(t)
        seen.add(key)
    return terms


def candidate_text_blob(result_obj: dict, task_name: str) -> str:
    if not isinstance(result_obj, dict):
        return ''
    texts = []
    for k, v in result_obj.items():
        if not isinstance(v, str):
            continue
        if '_base64_' in k:
            continue
        if (f'_{task_name}_' in k) or ('caption' in k.lower()) or ('visual_intent' in k.lower()):
            texts.append(v)
    return '\n'.join(texts)


def tokenize_text(text: str):
    return set(re.findall(r'[a-z0-9]{3,}', str(text or '').lower()))


def caption_alignment_score(candidate_blob: str, caption_text: str, visual_intent_text: str) -> float:
    ref_tokens = tokenize_text(caption_text) | tokenize_text(visual_intent_text)
    if not ref_tokens:
        return 0.0
    cand_tokens = tokenize_text(candidate_blob)
    if not cand_tokens:
        return 0.0
    overlap = len(ref_tokens & cand_tokens)
    return float(overlap / max(1, len(ref_tokens)))


def score_candidate(
    result_obj: dict,
    task_name: str,
    max_rounds: int,
    caption_text: str,
    visual_intent_text: str,
    required_terms: list[str],
    strict_required_terms: bool = False,
):
    final_key = pick_final_image_key(result_obj, task_name, int(max_rounds))
    if not final_key:
        return -1.0, {
            'final_key': None,
            'term_ratio': 0.0,
            'caption_alignment': 0.0,
            'critic_ratio': 0.0,
            'score': -1.0,
            'reason': 'missing_final_image',
        }

    blob = candidate_text_blob(result_obj, task_name)
    blob_l = blob.lower()
    missing_terms = []

    if required_terms:
        missing_terms = [t for t in required_terms if t.lower() not in blob_l]
        term_hits = len(required_terms) - len(missing_terms)
        term_ratio = term_hits / len(required_terms)
    else:
        term_ratio = 1.0

    alignment = caption_alignment_score(blob, caption_text, visual_intent_text)

    critic_found = 0
    for i in range(max(0, int(max_rounds))):
        k = f'target_{task_name}_critic_desc{i}'
        if isinstance(result_obj.get(k), str) and result_obj.get(k).strip():
            critic_found += 1
    critic_ratio = critic_found / max(1, int(max_rounds))

    if strict_required_terms and required_terms and missing_terms:
        return -1.0, {
            'final_key': final_key,
            'term_ratio': float(term_ratio),
            'caption_alignment': 0.0,
            'critic_ratio': float(critic_ratio),
            'score': -1.0,
            'missing_terms': missing_terms,
            'reason': 'missing_required_terms',
        }

    score = (0.55 * term_ratio) + (0.30 * alignment) + (0.15 * critic_ratio)

    return float(score), {
        'final_key': final_key,
        'term_ratio': float(term_ratio),
        'caption_alignment': float(alignment),
        'critic_ratio': float(critic_ratio),
        'score': float(score),
        'missing_terms': missing_terms,
    }


async def image_required_terms_score(
    result_obj: dict,
    task_name: str,
    max_rounds: int,
    required_terms: list[str],
    caption_text: str,
    visual_intent_text: str,
    model_name: str,
):
    final_key = pick_final_image_key(result_obj, task_name, int(max_rounds))
    if not final_key:
        return 0.0, 'missing_final_image'
    image_b64 = result_obj.get(final_key)
    if not image_b64:
        return 0.0, 'missing_final_image'
    if not required_terms:
        return 0.0, 'no_required_terms'

    required_str = ', '.join(required_terms)
    prompt = (
        'You are evaluating whether key concepts are visually explicit in a scientific diagram.\n'
        f'Required concepts/terms: {required_str}\n'
        f'Caption context: {caption_text}\n'
        f'Visual intent context: {visual_intent_text}\n\n'
        'Score from 0.0 to 1.0 where:\n'
        '- 1.0: required concepts are clearly and explicitly shown as visual mechanisms\n'
        '- 0.5: concepts are partially shown or mostly text-only\n'
        '- 0.0: concepts are absent or not visually clear\n\n'
        'Return ONLY a number between 0 and 1.'
    )

    contents = [
        {'type': 'text', 'text': prompt},
        {
            'type': 'image',
            'source': {
                'type': 'base64',
                'media_type': 'image/jpeg',
                'data': image_b64,
            },
        },
    ]

    try:
        response_list = await generation_utils.call_gemini_with_retry_async(
            model_name=model_name,
            contents=contents,
            config=types.GenerateContentConfig(
                temperature=0.0,
                candidate_count=1,
                max_output_tokens=64,
            ),
            max_attempts=2,
            retry_delay=4,
        )
    except Exception as exc:
        return 0.0, f'error:{type(exc).__name__}'

    raw = (response_list[0] or '').strip() if response_list else ''
    m = re.search(r'(?<!\\d)(0(?:\\.\\d+)?|1(?:\\.0+)?)(?!\\d)', raw)
    if not m:
        return 0.0, 'parse_failed'
    score = float(m.group(1))
    score = max(0.0, min(1.0, score))
    return score, 'ok'


def estimate_cost_preview(
    model_name: str,
    image_model_name: str,
    pipeline_mode: str,
    ui_mode: str,
    num_candidates: int,
    max_critic_rounds: int,
    method_text: str,
    visual_intent_text: str,
    figure_caption_text: str,
    input_mode: str,
    run_refinement: bool,
    refine_resolution: str,
):
    batch_size = int(num_candidates) if str(ui_mode).lower() == 'advanced' else 1
    ranges = pipeline_call_ranges(pipeline_mode, max_critic_rounds, batch_size)

    base_context_tokens = (
        approx_tokens(method_text)
        + approx_tokens(visual_intent_text)
        + approx_tokens(figure_caption_text)
    )

    # Prompt-size templates (token-level estimates) for planner/stylist/critic text calls.
    planner_in = base_context_tokens + 1200
    planner_out = 1400
    stylist_in = base_context_tokens + 1800
    stylist_out = 1100
    critic_in = base_context_tokens + 1600
    critic_out = 900

    text_input_min = (
        ranges['planner_calls'] * planner_in
        + ranges['stylist_calls'] * stylist_in
        + ranges['critic_calls_min'] * critic_in
    )
    text_input_max = (
        ranges['planner_calls'] * planner_in
        + ranges['stylist_calls'] * stylist_in
        + ranges['critic_calls_max'] * critic_in
    )

    text_output_min = (
        ranges['planner_calls'] * planner_out
        + ranges['stylist_calls'] * stylist_out
        + ranges['critic_calls_min'] * critic_out
    )
    text_output_max = (
        ranges['planner_calls'] * planner_out
        + ranges['stylist_calls'] * stylist_out
        + ranges['critic_calls_max'] * critic_out
    )

    # Step 8 auto-extraction estimate (if drive PDF auto mode was used).
    if str(input_mode).lower() == 'drive_pdf_auto':
        payload_chars = len(str(globals().get('candidate_text', '') or method_text or ''))
        payload_cap = int(globals().get('STEP8_MAX_PAYLOAD_CHARS', globals().get('STEP5C_MAX_PAYLOAD_CHARS', 32000)))
        bounded_chars = min(payload_chars, payload_cap)
        pass_a_in = approx_tokens('x' * bounded_chars) + 600
        pass_b_in = 2200
        pass_ab_out = 2600

        text_input_min += pass_a_in + pass_b_in
        text_output_min += pass_ab_out

        # One repair + one targeted regen upper-bound allowance.
        text_input_max += pass_a_in + pass_b_in + 3500
        text_output_max += pass_ab_out + 2200

    text_rates = model_rates_or_none(model_name)
    if text_rates:
        text_cost_min = (text_input_min / 1_000_000) * text_rates['input'] + (text_output_min / 1_000_000) * text_rates['output']
        text_cost_max = (text_input_max / 1_000_000) * text_rates['input'] + (text_output_max / 1_000_000) * text_rates['output']
    else:
        text_cost_min = None
        text_cost_max = None

    img_1k_rate, img_rate_fallback = image_output_rate_or_none(image_model_name, '1K')
    image_cost_min = None
    image_cost_max = None
    if img_1k_rate is not None:
        image_cost_min = ranges['image_calls_min'] * img_1k_rate
        image_cost_max = ranges['image_calls_max'] * img_1k_rate

    refinement_applies = bool(run_refinement and str(ui_mode).lower() == 'advanced')
    refinement_note = ''
    if refinement_applies:
        ref_rate, ref_fallback = image_output_rate_or_none(image_model_name, str(refine_resolution).upper())
        if ref_rate is not None and image_cost_min is not None:
            image_cost_min += ref_rate
            image_cost_max += ref_rate
            if ref_fallback:
                refinement_note = f'Refinement priced at 1K fallback for {refine_resolution} (model rate unavailable for that size).'
        ref_input_cost = IMAGE_INPUT_USD_PER_IMAGE.get(str(image_model_name).strip())
        if ref_input_cost and image_cost_min is not None:
            image_cost_min += ref_input_cost
            image_cost_max += ref_input_cost

    total_min = None
    total_max = None
    if (text_cost_min is not None) and (image_cost_min is not None):
        total_min = text_cost_min + image_cost_min
        total_max = text_cost_max + image_cost_max

    return {
        'batch_size': batch_size,
        'ranges': ranges,
        'text_input_min': text_input_min,
        'text_input_max': text_input_max,
        'text_output_min': text_output_min,
        'text_output_max': text_output_max,
        'text_cost_min': text_cost_min,
        'text_cost_max': text_cost_max,
        'image_cost_min': image_cost_min,
        'image_cost_max': image_cost_max,
        'total_min': total_min,
        'total_max': total_max,
        'image_rate_fallback': img_rate_fallback,
        'refinement_note': refinement_note,
    }


async def refine_selected_image_with_model(
    image_pil: Image.Image,
    prompt: str,
    image_model_name: str,
    aspect_ratio: str,
    image_size: str,
):
    img_buf = BytesIO()
    image_pil.save(img_buf, format='JPEG', quality=95)
    img_b64 = base64.b64encode(img_buf.getvalue()).decode('utf-8')

    contents = [
        {'type': 'text', 'text': prompt},
        {
            'type': 'image',
            'source': {
                'type': 'base64',
                'media_type': 'image/jpeg',
                'data': img_b64,
            },
        },
    ]

    response_list = await generation_utils.call_gemini_with_retry_async(
        model_name=image_model_name,
        contents=contents,
        config=types.GenerateContentConfig(
            temperature=1.0,
            candidate_count=1,
            max_output_tokens=8192,
            response_modalities=['IMAGE'],
            image_config=types.ImageConfig(
                aspect_ratio=aspect_ratio,
                image_size=image_size,
            ),
        ),
        max_attempts=4,
        retry_delay=8,
    )

    if not response_list or not response_list[0] or response_list[0].strip() == 'Error':
        return None

    try:
        refined_img = Image.open(BytesIO(base64.b64decode(response_list[0]))).convert('RGB')
        return refined_img
    except Exception:
        return None


task_name = str(TASK_NAME).strip().lower()
print('Starting Step 9 pipeline... this can take a few minutes.')
if task_name not in {'diagram', 'plot'}:
    raise ValueError(f'Unsupported TASK_NAME: {TASK_NAME}')

# Guardrails for retrieval modes requiring dataset
if RETRIEVAL_SETTING != 'none':
    ref_path = Path(REPO_DIR) / 'data' / 'PaperBananaBench' / task_name / 'ref.json'
    if not ref_path.exists():
        raise ValueError(
            f'RETRIEVAL_SETTING={RETRIEVAL_SETTING} requires dataset files. Missing: {ref_path}. '
            'Use RETRIEVAL_SETTING=none or prepare PaperBananaBench.'
        )

cost_preview = estimate_cost_preview(
    model_name=MODEL_NAME,
    image_model_name=IMAGE_MODEL_NAME,
    pipeline_mode=PIPELINE_MODE,
    ui_mode=UI_MODE,
    num_candidates=int(NUM_CANDIDATES),
    max_critic_rounds=int(MAX_CRITIC_ROUNDS),
    method_text=method_text,
    visual_intent_text=visual_intent_text,
    figure_caption_text=figure_caption_text,
    input_mode=INPUT_MODE,
    run_refinement=RUN_REFINEMENT,
    refine_resolution=REFINE_RESOLUTION,
)

print('\n=== Cost Preview (USD) ===')
print(f"Pricing source: {PRICING_REFERENCE_URL} (verified {PRICING_LAST_VERIFIED})")
print(f"Text model: {MODEL_NAME}")
print(f"Image model: {IMAGE_MODEL_NAME}")
print(f"Pipeline mode: {PIPELINE_MODE} | UI mode: {UI_MODE} | Batch size: {cost_preview['batch_size']}")

r = cost_preview['ranges']
print(
    'Estimated API call ranges -> '
    f"planner={r['planner_calls']}, stylist={r['stylist_calls']}, "
    f"critic={r['critic_calls_min']}-{r['critic_calls_max']}, "
    f"image={r['image_calls_min']}-{r['image_calls_max']}"
)

if cost_preview['text_cost_min'] is not None:
    print(
        f"Estimated text cost: ${cost_preview['text_cost_min']:.4f} - ${cost_preview['text_cost_max']:.4f} "
        f"(input ~{cost_preview['text_input_min']:,}-{cost_preview['text_input_max']:,} tok, "
        f"output ~{cost_preview['text_output_min']:,}-{cost_preview['text_output_max']:,} tok)"
    )
else:
    print('Estimated text cost: unavailable for this model ID (not in local pricing map).')

if cost_preview['image_cost_min'] is not None:
    print(
        f"Estimated image cost: ${cost_preview['image_cost_min']:.4f} - ${cost_preview['image_cost_max']:.4f} "
        '(includes image output pricing; refinement included when enabled)'
    )
else:
    print('Estimated image cost: unavailable for this model ID (not in local pricing map).')

if cost_preview['total_min'] is not None:
    print(f"Estimated total run cost: ${cost_preview['total_min']:.4f} - ${cost_preview['total_max']:.4f}")

if cost_preview['image_rate_fallback']:
    print('Note: used 1K image rate as fallback for image-cost estimate.')
if cost_preview['refinement_note']:
    print('Note:', cost_preview['refinement_note'])
print('Cost preview is an estimate; actual billing depends on final token/image usage.')

exp_config = config.ExpConfig(
    dataset_name='PaperBananaBench',
    task_name=task_name,
    split_name='colab',
    exp_mode=PIPELINE_MODE,
    retrieval_setting=RETRIEVAL_SETTING,
    max_critic_rounds=MAX_CRITIC_ROUNDS,
    model_name=MODEL_NAME,
    image_model_name=IMAGE_MODEL_NAME,
    work_dir=Path(REPO_DIR),
)

processor = PaperVizProcessor(
    exp_config=exp_config,
    vanilla_agent=VanillaAgent(exp_config=exp_config),
    planner_agent=PlannerAgent(exp_config=exp_config),
    visualizer_agent=VisualizerAgent(exp_config=exp_config),
    stylist_agent=StylistAgent(exp_config=exp_config),
    critic_agent=CriticAgent(exp_config=exp_config),
    retriever_agent=RetrieverAgent(exp_config=exp_config),
    polish_agent=PolishAgent(exp_config=exp_config),
)

run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir = OUTPUT_DIR / f'run_{task_name}_{run_stamp}'
run_dir.mkdir(parents=True, exist_ok=True)

# Expose these for Step 10
result = None
results = []
selected_result = None
selected_final_image = None
selected_final_key = None
active_task_name = task_name
refined_path = None

if UI_MODE == 'advanced':
    input_data_list = []
    for i in range(int(NUM_CANDIDATES)):
        item = {
            'id': f'colab_user_input_{i}',
            'filename': f'colab_user_input_{i}',
            'candidate_id': i,
            'caption': figure_caption_text.strip(),
            'content': method_text,
            'visual_intent': visual_intent_text.strip(),
            'additional_info': {'rounded_ratio': ASPECT_RATIO},
            'max_critic_rounds': int(MAX_CRITIC_ROUNDS),
        }
        input_data_list.append(item)

    async def run_batch(data_list):
        out = []
        async for result_data in processor.process_queries_batch(
            data_list,
            max_concurrent=int(MAX_CONCURRENT),
            do_eval=False,
        ):
            out.append(result_data)
        return out

    print(f'Generating {int(NUM_CANDIDATES)} candidates (advanced mode)...')
    results = await run_batch(input_data_list)
    results = sorted(results, key=lambda x: int(x.get('candidate_id', 0)))

    candidates_dir = run_dir / 'candidates'
    candidates_dir.mkdir(parents=True, exist_ok=True)

    candidate_images = []
    candidate_display_images = []
    candidate_artifacts = []
    w4k, h4k = target_4k_size(ASPECT_RATIO)

    for i, r in enumerate(results):
        key = pick_final_image_key(r, task_name, int(MAX_CRITIC_ROUNDS))

        candidate_json_path = candidates_dir / f'candidate_{i:02d}.json'
        candidate_json_payload = r if SAVE_FULL_RESULT_WITH_BASE64 else strip_base64_fields(r)
        candidate_json_path.write_text(json.dumps(candidate_json_payload, ensure_ascii=False, indent=2), encoding='utf-8')

        artifact = {
            'candidate_index': i,
            'final_key': key,
            'has_image': bool(key),
            'json_path': str(candidate_json_path),
            'raw_model_output_png_path': None,
            'final_deliverable_png_path': None,
        }

        if not key:
            candidate_images.append(None)
            candidate_display_images.append(None)
            candidate_artifacts.append(artifact)
            continue

        img = b64_to_image(r[key])
        candidate_images.append(img)

        raw_model_output_png_path = candidates_dir / f'candidate_{i:02d}_raw_model_output.png'
        img.save(raw_model_output_png_path, format='PNG', optimize=True)
        artifact['raw_model_output_png_path'] = str(raw_model_output_png_path)

        if EXPORT_4K:
            final_deliverable_img = img.resize((w4k, h4k), Image.Resampling.LANCZOS)
        else:
            final_deliverable_img = img

        final_deliverable_png_path = candidates_dir / f'candidate_{i:02d}_final_deliverable.png'
        final_deliverable_img.save(final_deliverable_png_path, format='PNG', optimize=True)

        artifact['final_deliverable_png_path'] = str(final_deliverable_png_path)
        candidate_display_images.append(final_deliverable_img)
        candidate_artifacts.append(artifact)

    display(Markdown(f'### Generated Candidates ({len(results)})'))
    n = len(candidate_images)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else axes

    for idx_ax, ax in enumerate(axes_flat):
        if idx_ax < n and candidate_images[idx_ax] is not None:
            ax.imshow(candidate_images[idx_ax])
            ax.set_title(f'Candidate {idx_ax}')
            ax.axis('off')
        elif idx_ax < n:
            ax.text(0.5, 0.5, f'Candidate {idx_ax}\nNo image', ha='center', va='center')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

    display(Markdown('### Final Deliverable: All Candidates'))
    for idx_candidate, cand_img in enumerate(candidate_display_images):
        if cand_img is None:
            continue
        display(Markdown(f'#### Candidate {idx_candidate}'))
        display(cand_img)

    required_terms_for_select = parse_required_terms(globals().get('MUST_INCLUDE_TERMS', ''))
    strict_required_terms = bool(globals().get('SELECTION_STRICT_REQUIRED_TERMS', True))
    image_aware_selection = bool(globals().get('IMAGE_AWARE_SELECTION', False))
    image_aware_topk = max(1, int(globals().get('IMAGE_AWARE_TOPK', 3)))
    image_aware_weight = float(globals().get('IMAGE_AWARE_WEIGHT', 0.35))
    image_aware_weight = max(0.0, min(1.0, image_aware_weight))
    image_aware_model = str(globals().get('IMAGE_AWARE_SELECTION_MODEL', '')).strip() or MODEL_NAME
    selection_mode = 'manual_index'
    score_cards = []

    if bool(globals().get('AUTO_SELECT_BEST', True)):
        for idx_candidate, cand in enumerate(results):
            cand_score, details = score_candidate(
                result_obj=cand,
                task_name=task_name,
                max_rounds=int(MAX_CRITIC_ROUNDS),
                caption_text=figure_caption_text,
                visual_intent_text=visual_intent_text,
                required_terms=required_terms_for_select,
                strict_required_terms=strict_required_terms,
            )
            score_cards.append({'candidate_index': idx_candidate, **details})

        valid_cards = [c for c in score_cards if c.get('score', -1.0) >= 0]
        if strict_required_terms and required_terms_for_select:
            valid_cards = [
                c for c in valid_cards
                if float(c.get('term_ratio', 0.0)) >= 0.999999
            ]
        if valid_cards:
            if image_aware_selection and required_terms_for_select:
                sorted_cards = sorted(
                    valid_cards,
                    key=lambda c: (c.get('score', -1.0), c.get('term_ratio', 0.0), c.get('caption_alignment', 0.0)),
                    reverse=True,
                )
                top_cards = sorted_cards[: min(image_aware_topk, len(sorted_cards))]
                for c in valid_cards:
                    c['final_selection_score'] = float(c.get('score', -1.0))
                    c['image_semantic_score'] = None
                    c['image_semantic_reason'] = 'not_reranked'
                for c in top_cards:
                    idx_candidate = int(c['candidate_index'])
                    image_score, image_reason = await image_required_terms_score(
                        result_obj=results[idx_candidate],
                        task_name=task_name,
                        max_rounds=int(MAX_CRITIC_ROUNDS),
                        required_terms=required_terms_for_select,
                        caption_text=figure_caption_text,
                        visual_intent_text=visual_intent_text,
                        model_name=image_aware_model,
                    )
                    c['image_semantic_score'] = float(image_score)
                    c['image_semantic_reason'] = image_reason
                    c['final_selection_score'] = float((1.0 - image_aware_weight) * float(c.get('score', 0.0)) + image_aware_weight * float(image_score))
                best = max(valid_cards, key=lambda c: (c.get('final_selection_score', c.get('score', -1.0)), c.get('term_ratio', 0.0), c.get('caption_alignment', 0.0)))
                selected_idx = int(best['candidate_index'])
                selection_mode = 'auto_score_image_aware'
                print(
                    f"Auto-selected candidate {selected_idx} with image-aware rerank "
                    f"(final={best.get('final_selection_score', best.get('score', 0.0)):.3f}, text={best.get('score', 0.0):.3f}, "
                    f"img={float(best.get('image_semantic_score', 0.0)):.3f})"
                )
            else:
                best = max(valid_cards, key=lambda c: (c['score'], c['term_ratio'], c['caption_alignment']))
                selected_idx = int(best['candidate_index'])
                selection_mode = 'auto_score'
                print(
                    f"Auto-selected candidate {selected_idx} (score={best['score']:.3f}, "
                    f"term_ratio={best['term_ratio']:.3f}, caption_alignment={best['caption_alignment']:.3f})"
                )
        else:
            selected_idx = int(SELECTED_CANDIDATE_INDEX)
            print('Auto-select could not find a valid candidate under current constraints; falling back to SELECTED_CANDIDATE_INDEX.')
    else:
        selected_idx = int(SELECTED_CANDIDATE_INDEX)

    if selected_idx < 0:
        selected_idx = 0
    if selected_idx >= len(results):
        selected_idx = len(results) - 1

    selected_result = results[selected_idx]
    result = selected_result
    selected_final_key = pick_final_image_key(selected_result, task_name, int(MAX_CRITIC_ROUNDS))
    if not selected_final_key:
        raise RuntimeError('Selected candidate has no output image.')

    selected_final_image = b64_to_image(selected_result[selected_final_key])

    selected_dir = run_dir / 'selected'
    selected_dir.mkdir(parents=True, exist_ok=True)

    selected_raw_model_output_path = selected_dir / f'selected_raw_model_output_{task_name}.jpg'
    selected_final_image.save(selected_raw_model_output_path, format='JPEG', quality=95)

    all_results_path = run_dir / 'all_results.json'
    selected_result_path = selected_dir / 'selected_result.json'
    results_json_payload = results if SAVE_FULL_RESULT_WITH_BASE64 else strip_base64_fields(results)
    selected_json_payload = selected_result if SAVE_FULL_RESULT_WITH_BASE64 else strip_base64_fields(selected_result)
    all_results_path.write_text(json.dumps(results_json_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    selected_result_path.write_text(json.dumps(selected_json_payload, ensure_ascii=False, indent=2), encoding='utf-8')

    if EXPORT_4K:
        w4k, h4k = target_4k_size(ASPECT_RATIO)
        selected_4k = selected_final_image.resize((w4k, h4k), Image.Resampling.LANCZOS)
    else:
        selected_4k = selected_final_image
    selected_final_deliverable_path = selected_dir / f'selected_final_deliverable_{task_name}.png'
    selected_4k.save(selected_final_deliverable_path, format='PNG', optimize=True)

    display(Markdown(f'### Selected Candidate: {selected_idx}'))
    display(Markdown('### Final Deliverable: Selected Candidate'))
    display(selected_4k)

    refined_path = None
    if RUN_REFINEMENT:
        if REFINE_SOURCE == 'external_image_path':
            external_path = Path(EXTERNAL_REFINE_IMAGE_PATH).expanduser()
            if not external_path.exists():
                raise FileNotFoundError(
                    f'EXTERNAL_REFINE_IMAGE_PATH does not exist: {external_path}. '
                    'Set REFINE_SOURCE=selected_result or provide a valid image path.'
                )
            refine_input_image = Image.open(external_path).convert('RGB')
        else:
            refine_input_image = selected_final_image

        refined_img = await refine_selected_image_with_model(
            image_pil=refine_input_image,
            prompt=REFINE_PROMPT,
            image_model_name=IMAGE_MODEL_NAME,
            aspect_ratio=REFINE_ASPECT_RATIO,
            image_size=REFINE_RESOLUTION,
        )
        if refined_img is not None:
            refined_path = selected_dir / f'selected_refined_{REFINE_RESOLUTION}.png'
            refined_img.save(refined_path, format='PNG', optimize=True)
            display(Markdown(f'### Refined Selected Image ({REFINE_RESOLUTION})'))
            display(refined_img)
        else:
            print('Refinement failed or model did not return an image.')

    resolved_inputs = {
        'task_name': task_name,
        'ui_mode': UI_MODE,
        'input_mode': INPUT_MODE,
        'resolved_input_path': resolved_input_path,
        'visual_intent_text': visual_intent_text,
        'figure_caption_text': figure_caption_text,
        'source_context_chars': len(method_text),
        'pipeline_mode': PIPELINE_MODE,
        'retrieval_setting': RETRIEVAL_SETTING,
        'num_candidates': int(NUM_CANDIDATES),
        'max_concurrent': int(MAX_CONCURRENT),
        'selected_candidate_index': selected_idx,
        'selected_final_key': selected_final_key,
        'selection_mode': selection_mode,
        'auto_select_best': bool(globals().get('AUTO_SELECT_BEST', True)),
        'selection_strict_required_terms': bool(globals().get('SELECTION_STRICT_REQUIRED_TERMS', True)),
        'image_aware_selection': bool(globals().get('IMAGE_AWARE_SELECTION', False)),
        'image_aware_topk': int(globals().get('IMAGE_AWARE_TOPK', 3)),
        'image_aware_weight': float(globals().get('IMAGE_AWARE_WEIGHT', 0.35)),
        'image_aware_selection_model': str(globals().get('IMAGE_AWARE_SELECTION_MODEL', '')).strip() or MODEL_NAME,
        'candidate_scores': score_cards,
        'candidate_artifacts': candidate_artifacts,
        'candidates_dir': str(candidates_dir),
        'context_scope': str(globals().get('STEP8_CONTEXT_SCOPE_USED', globals().get('CONTEXT_SCOPE', 'unknown'))),
        'must_include_terms': parse_required_terms(globals().get('MUST_INCLUDE_TERMS', '')),
        'step8_coverage_score': float(globals().get('STEP8_COVERAGE_SCORE', 0.0)),
        'step8_missing_terms': list(globals().get('STEP8_MISSING_TERMS', [])),
        'export_4k': bool(EXPORT_4K),
        'run_refinement': bool(RUN_REFINEMENT),
        'refine_source': REFINE_SOURCE,
        'external_refine_image_path': EXTERNAL_REFINE_IMAGE_PATH,
        'refine_resolution': REFINE_RESOLUTION,
        'refine_aspect_ratio': REFINE_ASPECT_RATIO,
    }
    resolved_inputs_path = run_dir / 'resolved_inputs.json'
    resolved_inputs_path.write_text(json.dumps(resolved_inputs, ensure_ascii=False, indent=2), encoding='utf-8')

    if ZIP_EXPORT:
        zip_path = run_dir / 'export_bundle.zip'
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for file_path in sorted(run_dir.rglob('*')):
                if file_path.is_file() and file_path.name != 'export_bundle.zip':
                    zf.write(file_path, arcname=str(file_path.relative_to(run_dir)))
        print('Saved ZIP bundle:', zip_path)

    print('Advanced run completed:', run_dir)
    print('Saved all results:', all_results_path)
    print('Saved candidate artifacts dir:', candidates_dir)
    print('Saved selected final deliverable:', selected_final_deliverable_path)
    print(f'Saved candidate final deliverables: {sum(1 for x in candidate_artifacts if x.get("final_deliverable_png_path"))}')
    if not SAVE_FULL_RESULT_WITH_BASE64:
        print('Saved slim JSON (base64 fields removed for smaller files).')

else:
    input_sample = {
        'id': 'colab_user_input',
        'filename': 'colab_user_input',
        'caption': figure_caption_text.strip(),
        'content': method_text,
        'visual_intent': visual_intent_text.strip(),
        'additional_info': {'rounded_ratio': ASPECT_RATIO},
        'max_critic_rounds': int(MAX_CRITIC_ROUNDS),
    }

    async def run_one(sample):
        outputs = []
        async for result_data in processor.process_queries_batch(
            [sample],
            max_concurrent=1,
            do_eval=False,
        ):
            outputs.append(result_data)
        return outputs[0]

    print('Running simple mode generation (expected ~2-5 minutes)...')
    result = await run_one(input_sample)
    selected_result = result

    final_image_key = pick_final_image_key(result, task_name, int(MAX_CRITIC_ROUNDS))
    if not final_image_key:
        raise RuntimeError('No output image found. Check model names, API key, and input quality.')

    final_image = b64_to_image(result[final_image_key])

    raw_model_output_path = run_dir / f'final_{task_name}_raw_model_output.jpg'
    final_deliverable_path = run_dir / f'final_{task_name}_final_deliverable.png'
    result_json_path = run_dir / 'full_result.json'
    resolved_inputs_path = run_dir / 'resolved_inputs.json'
    description_path = run_dir / f'final_{task_name}_description.txt'

    final_image.save(raw_model_output_path, format='JPEG', quality=95)

    if EXPORT_4K:
        w4k, h4k = target_4k_size(ASPECT_RATIO)
        final_4k = final_image.resize((w4k, h4k), Image.Resampling.LANCZOS)
    else:
        final_4k = final_image

    final_4k.save(final_deliverable_path, format='PNG', optimize=True)
    result_json_payload = result if SAVE_FULL_RESULT_WITH_BASE64 else strip_base64_fields(result)
    result_json_path.write_text(json.dumps(result_json_payload, ensure_ascii=False, indent=2), encoding='utf-8')

    resolved_inputs = {
        'task_name': task_name,
        'ui_mode': UI_MODE,
        'input_mode': INPUT_MODE,
        'resolved_input_path': resolved_input_path,
        'visual_intent_text': visual_intent_text,
        'figure_caption_text': figure_caption_text,
        'source_context_chars': len(method_text),
        'pipeline_mode': PIPELINE_MODE,
        'retrieval_setting': RETRIEVAL_SETTING,
        'context_scope': str(globals().get('STEP8_CONTEXT_SCOPE_USED', globals().get('CONTEXT_SCOPE', 'unknown'))),
        'must_include_terms': parse_required_terms(globals().get('MUST_INCLUDE_TERMS', '')),
        'step8_coverage_score': float(globals().get('STEP8_COVERAGE_SCORE', 0.0)),
        'step8_missing_terms': list(globals().get('STEP8_MISSING_TERMS', [])),
        'export_4k': bool(EXPORT_4K),
        'final_deliverable_path': str(final_deliverable_path),
        'final_4k_path': str(final_deliverable_path),
    }
    resolved_inputs_path.write_text(json.dumps(resolved_inputs, ensure_ascii=False, indent=2), encoding='utf-8')

    desc_key = final_image_key.replace('_base64_jpg', '')
    final_description = result.get(desc_key, '')
    if final_description:
        description_path.write_text(final_description, encoding='utf-8')

    print('Simple run completed:', run_dir)
    print('Saved raw model output:', raw_model_output_path)
    print('Saved final deliverable:', final_deliverable_path)
    print('Saved result JSON:', result_json_path)
    if not SAVE_FULL_RESULT_WITH_BASE64:
        print('Saved slim JSON (base64 fields removed for smaller files).')

    display(Markdown('### Final Deliverable'))
    display(final_4k)

if selected_result is None:
    selected_result = result

observed_results = results if isinstance(results, list) and results else ([selected_result] if isinstance(selected_result, dict) else [])
observed_image_outputs = sum(count_image_keys(x, task_name) for x in observed_results)

obs_image_cost = None
obs_1k_rate, _ = image_output_rate_or_none(IMAGE_MODEL_NAME, '1K')
if obs_1k_rate is not None:
    obs_image_cost = observed_image_outputs * obs_1k_rate

# Advanced-mode optional refinement call (one extra output image if it succeeded).
if (UI_MODE == 'advanced') and bool(RUN_REFINEMENT) and (refined_path is not None):
    ref_rate, _ = image_output_rate_or_none(IMAGE_MODEL_NAME, str(REFINE_RESOLUTION).upper())
    if (ref_rate is None) and (obs_1k_rate is not None):
        ref_rate = obs_1k_rate
    if (obs_image_cost is not None) and (ref_rate is not None):
        obs_image_cost += ref_rate
    ref_input_cost = IMAGE_INPUT_USD_PER_IMAGE.get(str(IMAGE_MODEL_NAME).strip())
    if (obs_image_cost is not None) and ref_input_cost:
        obs_image_cost += ref_input_cost

print('\n=== Observed Cost Summary (from generated outputs) ===')
print(f'Observed generated images: {observed_image_outputs}')
if obs_image_cost is not None:
    print(f'Observed image-cost estimate: ${obs_image_cost:.4f} (based on generated image count)')
else:
    print('Observed image-cost estimate unavailable for this image model ID.')
print('Text token billing is shown in the pre-run estimate range above.')



In [ ]:
#@title Step 10 (optional): Show intermediate stages
import base64
from io import BytesIO

import matplotlib.pyplot as plt
from PIL import Image

SHOW_STAGE_DESCRIPTIONS = globals().get('SHOW_STAGE_DESCRIPTIONS', True)
SHOW_CRITIC_SUGGESTIONS = globals().get('SHOW_CRITIC_SUGGESTIONS', True)

STAGE_VIEW_CANDIDATE_INDEX = -1  # @param {type:"integer"}

source_result = None
if 'results' in globals() and isinstance(results, list) and len(results) > 0 and STAGE_VIEW_CANDIDATE_INDEX >= 0:
    idx = min(max(int(STAGE_VIEW_CANDIDATE_INDEX), 0), len(results) - 1)
    source_result = results[idx]
    print(f'Using candidate {idx} for stage view.')
elif 'selected_result' in globals() and selected_result is not None:
    source_result = selected_result
    print('Using selected result for stage view.')
elif 'result' in globals() and result is not None:
    source_result = result
    print('Using single result for stage view.')
else:
    raise ValueError('No generation result found. Run Step 9 first.')

task_name = str(globals().get('active_task_name', globals().get('TASK_NAME', 'diagram'))).strip().lower()

stage_order = [
    ('Planner', f'target_{task_name}_desc0_base64_jpg', f'target_{task_name}_desc0', None),
    ('Stylist', f'target_{task_name}_stylist_desc0_base64_jpg', f'target_{task_name}_stylist_desc0', None),
    ('Critic Round 0', f'target_{task_name}_critic_desc0_base64_jpg', f'target_{task_name}_critic_desc0', f'target_{task_name}_critic_suggestions0'),
    ('Critic Round 1', f'target_{task_name}_critic_desc1_base64_jpg', f'target_{task_name}_critic_desc1', f'target_{task_name}_critic_suggestions1'),
    ('Critic Round 2', f'target_{task_name}_critic_desc2_base64_jpg', f'target_{task_name}_critic_desc2', f'target_{task_name}_critic_suggestions2'),
]

shown = 0
for stage_name, image_key, desc_key, sugg_key in stage_order:
    image_b64 = source_result.get(image_key)
    if not image_b64:
        continue

    img = Image.open(BytesIO(base64.b64decode(image_b64))).convert('RGB')
    plt.figure(figsize=(12, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'{stage_name} ({image_key})')
    plt.show()

    if SHOW_STAGE_DESCRIPTIONS and source_result.get(desc_key):
        print(f'[{stage_name}] Description:')
        print(source_result[desc_key])
        print()

    if task_name == 'plot' and SHOW_STAGE_DESCRIPTIONS and source_result.get(f'{desc_key}_code'):
        print(f'[{stage_name}] Generated matplotlib code:')
        print(source_result[f'{desc_key}_code'])
        print()

    if SHOW_CRITIC_SUGGESTIONS and sugg_key and source_result.get(sugg_key):
        print(f'[{stage_name}] Critic suggestions:')
        print(source_result[sugg_key])
        print()

    shown += 1

if shown == 0:
    print('No intermediate stage images were available for this result.')
else:
    print('Done.')


## FAQ

- What is this notebook optimized for?
  - A Colab-first, Google API key workflow for non-coders.
  - Goal: paste key, provide file/text, run end-to-end in 10 steps.

- Step 4 fails with API key errors. What should I do?
  - Verify your key at https://aistudio.google.com/apikey.
  - Re-run Step 4 after updating `GOOGLE_API_KEY_INPUT`.

- I added OPENAI/ANTHROPIC keys. Does the default flow change?
  - No. Default flow stays Gemini text + Nano Banana image presets.
  - OpenAI key is used only for compatible custom branches (for example `gpt-image-*` image models).
  - Anthropic key is loaded for compatibility, but not used by the default notebook path.
  - Keep `STORE_API_KEY_IN_CONFIG=False` (default) so keys are not written to Drive config.

- I cannot find the upload button in Step 5.
  - Set `FILE_INPUT_METHOD='upload_from_computer'` and re-run Step 5.

- Step 8 says input file not found.
  - Upload to the `INPUT_DIR` path printed in Step 1 (default: `MyDrive/PaperBanana/input/`).
  - Update `FALLBACK_SELECTED_PATH` or `PLOT_RAW_DATA_FILE_PATH`.

- Step 8 times out or fails repeatedly.
  - Keep `TEXT_MODEL_PRESET='fast_2_5_flash'` in Step 4.
  - Keep `STEP8_FALLBACK_MODELS='gemini-2.5-pro,gemini-3.1-pro-preview'` in Step 7.
  - Increase `STEP8_PER_ATTEMPT_TIMEOUT` if needed.

- Step 9 is slow even in simple mode.
  - Keep `STYLE_MODE='planner_critic_fast'` in Step 6 (default).
  - Set `MAX_CRITIC_ROUNDS=1` in Step 7.
  - Keep `RUN_REFINEMENT=False`, `EXPORT_4K=False`, and `ZIP_EXPORT=False` for fastest runs.

- The advanced run is slow or expensive.
  - Reduce `NUM_CANDIDATES`, `MAX_CONCURRENT`, and `MAX_CRITIC_ROUNDS`.

- Plot output quality is weak.
  - Provide clearer `PLOT_VISUAL_INTENT` and clean, structured raw data.

- Runtime restarted and variables disappeared.
  - Re-run Steps 1 through 9 in order.

- Which model settings are safest for first run?
  - Keep Step 4 defaults, especially `TEXT_MODEL_PRESET='fast_2_5_flash'` and key validation enabled.
  - This default is reliability-first for Colab (helps reduce timeout/503 retry loops).
  - After a stable run, switch to `balanced_2_5_pro` (middle ground) or `paper_style_updated` (best quality, slower/costlier).


- Why does Step 6 use `STYLE_MODE` names instead of exact Streamlit pipeline names?
  - `STYLE_MODE` is a simplified alias for non-coder use in Step 6.
  - Mapping is: `planner_critic_fast -> demo_planner_critic`, `stylist_enhanced -> demo_full`.
  - If you want exact names, use `PIPELINE_MODE` in Step 7.

- Which input mode should I use?
  - Start with `drive_pdf_auto`.
  - Use `paste_text_manual` when you need strict manual control.

- How do I force specific terms to appear in the figure?
  - Set `MUST_INCLUDE_TERMS` in Step 6 (comma-separated), then re-run Step 8.

- Why can Step 8 miss specific concepts?
  - If `CONTEXT_SCOPE='method_only'` and terms are not locked, details can be generalized.
  - Use `CONTEXT_SCOPE='full_paper'` + `MUST_INCLUDE_TERMS` for stronger coverage.

- What is the difference between `raw_model_output` and `final_deliverable`?
  - `raw_model_output`: direct model image output.
  - `final_deliverable`: standardized output (4K when `EXPORT_4K=True`).

- In advanced mode, are all candidates saved?
  - Yes. Step 9 saves per-candidate JSON and images under `run_*/candidates/`.

- What does `STAGE_VIEW_CANDIDATE_INDEX` do?
  - In Step 10, `-1` shows the selected result; `0+` shows that candidate index from advanced mode.

